In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [0]:
"""
Multi-Dataset Join – Customer Journey Insights
Datasets:
users(user_id, region)
products(product_id, category)
events(user_id, product_id, event_type, event_time)
(event_type: view, click, purchase)

Tasks:
For each user:
Find average time taken between view → click → purchase (per product).
Rank categories by conversion speed in each region.
Flag users with high dropout after click (clicked but never purchased).
"""

In [0]:
users_data = [
    (1, "North"),
    (2, "South"),
    (3, "East"),
    (4, "West"),
    (5, "North")
]

users_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("region", StringType(), True)
])

users_df = spark.createDataFrame(users_data, schema=users_schema)
users_df.show()

products_data = [
    (101, "Electronics"),
    (102, "Fashion"),
    (103, "Home"),
    (104, "Fashion"),
    (105, "Electronics")
]

products_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("category", StringType(), True)
])

products_df = spark.createDataFrame(products_data, schema=products_schema)
products_df.show()


events_data = [
    (1, 101, "view",     "2024-06-01 09:00:00"),
    (1, 101, "click",    "2024-06-01 09:01:00"),
    (1, 101, "purchase", "2024-06-01 09:05:00"),

    (2, 102, "view",     "2024-06-01 10:00:00"),
    (2, 102, "click",    "2024-06-01 10:10:00"),
    # no purchase

    (3, 103, "view",     "2024-06-01 11:00:00"),
    # no click or purchase

    (4, 104, "view",     "2024-06-01 08:30:00"),
    (4, 104, "click",    "2024-06-01 08:35:00"),
    (4, 104, "purchase", "2024-06-01 08:45:00"),

    (5, 105, "view",     "2024-06-01 12:00:00"),
    (5, 105, "click",    "2024-06-01 12:05:00"),
    # no purchase
]

events_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_time", StringType(), True)
])

events_df = spark.createDataFrame(events_data, schema=events_schema)
events_df = events_df.withColumn(
    "event_time",
    to_timestamp("event_time")
)
events_df.show()

In [0]:
# For each user Find average time taken between view → click → purchase (per product).

# Window specification to get previous event time
window_spec = Window.partitionBy("user_id", "product_id").orderBy("event_time")

user_products_flow_df = events_df.withColumn(
    "prev_event_time",
    lag("event_time").over(window_spec)
)

# filtering null records
user_products_flow_df = user_products_flow_df.filter(col("prev_event_time").isNotNull())

# Checking the difference between event times
event_diff_df = user_products_flow_df.withColumn(
    "time_taken",
    (unix_timestamp(col("event_time")) - unix_timestamp(col("prev_event_time"))) /60
)

# Aggregating average time takens for products
products_avg_time_taken_df =event_diff_df.groupBy("product_id").agg(
    avg("time_taken").alias("avg_time_taken")
)
products_avg_time_taken_df.show()

In [0]:
# Rank categories by conversion speed in each region.

# Filtering only view and purchase events for products
filtered_events_df = events_df.filter(col("event_type").isin("view", "purchase"))

# Joining with users and products to get user region and product category
events_with_users_region_df = filtered_events_df.join(users_df, on="user_id", how="inner")

events_with_users_region_cat_df = events_with_users_region_df.join(products_df, on="product_id", how="inner")

# Window spec to get view time
window_spec = Window.partitionBy("user_id", "product_id").orderBy("event_time")

user_event_analysis_df = events_with_users_region_cat_df.withColumn(
    "view_time",
    lag("event_time").over(window_spec)
)
user_event_analysis_df = user_event_analysis_df.filter(col("view_time").isNotNull())
user_event_analysis_df = user_event_analysis_df.withColumnRenamed("event_time", "purchase_time")

products_conversion_speed_df = user_event_analysis_df.withColumn(
    "conversion_speed",
    (unix_timestamp(col("purchase_time")) - unix_timestamp(col("view_time"))) / 60
)
# products_conversion_speed_df.show()

avg_con_per_cate_region_df =products_conversion_speed_df.groupBy("region", "category").agg(
    avg("conversion_speed").alias("avg_con_speed")
)

# Ranking categories in region based on average conversion speed
window_rank = Window.partitionBy("region").orderBy(desc("avg_con_speed"))

category_ranking_df = avg_con_per_cate_region_df.withColumn(
    "rnk",
    dense_rank().over(window_rank)
)
category_ranking_df.show()

In [0]:
# Flag users with high dropout after click (clicked but never purchased).

# Window specification to get next event type
window_spec = Window.partitionBy("user_id", "product_id").orderBy("event_time")

users_product_details_df = events_df.withColumn(
    "next_event_type",
    lead("event_type").over(window_spec)
)

users_product_details_df = users_product_details_df.withColumn(
    "isNotPurchased",
    when((col("event_type") == "click") & (col("next_event_type").isNull()), 1).otherwise(0)
)

users_dropouts_df = users_product_details_df.groupBy("user_id").agg(
    count(when(col("isNotPurchased") == 1, True)).alias("total_dropouts")
)

users_droppouts_flags_df = users_dropouts_df.withColumn(
    "is_high_dropout",
    when(col("total_dropouts") >= 3, "Yes").otherwise("No")
)

users_droppouts_flags_df.show()

In [0]:
"""
Cumulative Product Supply Tracker
Dataset:
supplies(product_id, supplier_id, supply_date, units_supplied)

Tasks:
For each product:
Compute running total of supplied units over time (by date).
Identify products that were out of stock for 3+ consecutive supply days.
Rank suppliers by total units supplied and consistency (stddev of units/day).
"""

In [0]:
# Define schema
schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("supplier_id", StringType(), True),
    StructField("supply_date", StringType(), True),
    StructField("units_supplied", IntegerType(), True)
])

# Sample data
data = [
    ("P1", "S1", "2024-01-01", 10),
    ("P1", "S2", "2024-01-02", 0),
    ("P1", "S1", "2024-01-03", 0),
    ("P1", "S2", "2024-01-04", 0),
    ("P1", "S1", "2024-01-05", 15),
    ("P2", "S1", "2024-01-01", 5),
    ("P2", "S2", "2024-01-02", 10),
    ("P2", "S1", "2024-01-03", 0),
    ("P2", "S2", "2024-01-04", 0),
    ("P2", "S2", "2024-01-05", 20),
    ("P3", "S3", "2024-01-01", 0),
    ("P3", "S3", "2024-01-02", 0),
    ("P3", "S3", "2024-01-03", 0),
    ("P3", "S3", "2024-01-04", 30),
]

df = spark.createDataFrame(data, schema)

df = df.withColumn(
    "supply_date",
    to_date("supply_date")
)
df.show()

In [0]:
class  ProductSupplyTracker:
    def read_data(self,path):
        try:
            df = spark.read.format('csv').load(path)
            return df
        except Exception as err:
            print(f"Error caught while reading {path}: {err}")
            return None
        
    def productRunningTotals(df):
        # Window specification to aggregate running total for products
        window_spec = Window.partitionBy("product_id").orderBy("supply_date").rowsBetween(Window.unboundedPreceding, 0)
        # Calculating running totals for products
        products_running_totals_df = df.withColumn(
            "running_total",
            sum("units_supplied").over(window_spec)
        )    

        return products_running_totals_df
    
    def consequtiveOutOfStocks(df):
        # Assigning random values based on units
        df = df.withColumn(
            "is_out_of_stock",
            when(col("units_supplied") == 0, 1).otherwise(0)
        )
        # Assigning row numbers for products in a flow
        window_spec = Window.partitionBy("product_id").orderBy("supply_date")
        updated_df = df.withColumn(
            "rn",
            row_number().over(window_spec)
        )
        # Identifying streaks for products
        supp_streaks_df = updated_df.withColumn(
            "streak_group",
            expr("date_sub(supply_date, rn)")
        )
        # Aggregating out of stock days for each streak
        aggregated_df = supp_streaks_df.groupBy("product_id", "streak_group").agg(
            sum("is_out_of_stock").alias("total_consequtive_days")
        )
        # Filtering product streaks where cons days >= 3
        filtered_out_of_stocks_df = aggregated_df.filter(col("total_consequtive_days") >= 3)

        return filtered_out_of_stocks_df
    
    def rankingSuppliers(df):
        # Aggregating total units sold for each supplier
        supplier_daily_df = df.groupBy("supplier_id", "supply_date").agg(
            sum("units_supplied").alias("daily_units")
        )

        # Aggregating standard deviation for suppliers
        supplier_stddev_df = supplier_daily.groupBy("supplier_id").agg(
            stddev("daily_units").alias("units_stddev"),
            sum("daily_units").alias("total_units")
        )

        # Ranking suppliers based on total units and stddev per day
        window_rank = Window.orderBy(desc("total_units"))
        window_rank_two = Window.orderBy("units_stddev")

        ranked_by_total_df = supplier_stddev_df.withColumn(
            "rank_by_total_units",
            dense_rank().over(window_rank)
        )

        ranked_by_consistency_df = ranked_by_total_df.withColumn(
            "rnk",
            dense_rank().over(window_rank_two)
        )

        return ranked_by_total_df, ranked_by_consistency_df
    

# Instantiating the class
supply_track_inst = ProductSupplyTracker()

# Reading file as dataframe
suppliers_data_df = supply_track_inst.read_data("/Volumes/db_prac/prac/raw/suppliers_data")

# Calling methods
products_running_totals_df = supply_track_inst.productRunningTotals(suppliers_data_df)
filtered_out_of_stocks_df = supply_track_inst.consequtiveOutOfStocks(suppliers_data_df)
ranked_by_total_df, ranked_by_consistency_df = supply_track_inst.rankingSuppliers(suppliers_data_df)

# Showing dataframes
products_running_totals_df.show()
filtered_out_of_stocks_df.show()
ranked_by_total_df.show()
ranked_by_consistency_df.show()

In [0]:
"""
Customer Payment Behavior
Datasets:

payments(payment_id, customer_id, payment_date, amount)
invoices(invoice_id, customer_id, invoice_date, due_date, amount_due)

Tasks:
Join to find payments made against invoices (same customer, payment after invoice date).
Calculate:
Avg payment delay (days between invoice and payment).
Flag customers who paid after due date more than 50% of the time.
Aggregate unpaid amount per customer.
"""

In [0]:
invoices_schema = StructType([
    StructField("invoice_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("invoice_date", StringType(), True),
    StructField("due_date", StringType(), True),
    StructField("amount_due", DoubleType(), True)
])

invoices_data = [
    (1001, 101, "2024-06-05", "2024-06-12", 200.0),
    (1002, 101, "2024-06-15", "2024-06-22", 100.0),
    (1003, 102, "2024-06-25", "2024-07-02", 300.0),
    (1004, 103, "2024-06-10", "2024-06-18", 150.0),
    (1005, 104, "2024-06-30", "2024-07-04", 400.0),
    (1006, 105, "2024-06-20", "2024-06-27", 250.0),
]

invoices_df_raw = spark.createDataFrame(data=invoices_data, schema=invoices_schema)

invoices_df = invoices_df_raw \
    .withColumn("invoice_date", to_date("invoice_date", "yyyy-MM-dd")) \
    .withColumn("due_date", to_date("due_date", "yyyy-MM-dd"))

invoices_df.show()

payments_schema = StructType([
    StructField("payment_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("invoice_id", IntegerType(), True),
    StructField("payment_date", StringType(), True),
    StructField("amount", DoubleType(), True)
])

payments_data = [
    (1, 101, 1001, "2024-06-10", 200.0),  # on time
    (2, 101, 1002, "2024-06-25", 100.0),  # late
    (3, 102, 1003, "2024-07-01", 300.0),  # on time
    (4, 103, 1004, "2024-06-20", 150.0),  # late
    (5, 104, 1005, "2024-07-03", 400.0)  # on time
]

payments_df_raw = spark.createDataFrame(data=payments_data, schema=payments_schema)

payments_df = payments_df_raw.withColumn("payment_date", to_date("payment_date", "yyyy-MM-dd"))

payments_df.show()


In [0]:
class CustomerPaymentBehaviour:
    def read_data(self, path):
        try:
            df = spark.read.format('csv').option("header", True).load(path)
            return df
        except Exception as err:
            print(f"Error occur while reading file {path} : {err}")
            return None
        
    def invoices_with_payments(self, payments_df, invoices_df):
        invoices_with_payments_df = invoices_df.join(
            payments_df, 
            (invoices_df["customer_id"] == payments_df["customer_id"]) &
            (invoices_df["invoice_id"] == payments_df["invoice_id"]) &
            (payments_df["payment_date"] > invoices_df["invoice_date"]), 
            "inner"
        )

        return invoices_with_payments_df

    def avgPaymentDelay(self, payments_df, invoices_df):
        # Calling the joining function
        invoices_with_payments_df = self.invoices_with_payments(payments_df, invoices_df)
        
        # Checking the days difference between payment and due dates
        invoices_with_payments_df = invoices_with_payments_df.withColumn(
            "delay_diff",
            datediff(col("payment_date"), col("due_date"))
        )

        # Aggregating average delay for each customer
        cust_avg_delay_df = invoices_with_payments_df.groupBy("customer_id").agg(
            avg("delay_diff").alias("avg_delay")
        )

        return cust_avg_delay_df
    
    def flagingCustomers(self, payments_df, invoices_df):
        # Calling the joining function
        invoices_with_payments_df = self.invoices_with_payments(payments_df, invoices_df)

        # Assigning values where payment date > due date for invoices payments
        updated_df = invoices_with_payments_df.withColumn(
            "is_delayed",
            when(col("payment_date") > col("due_date"), 1).otherwise(0)
        )

        # Aggregating total paymets and delay payments counts for customers
        cust_delay_total_counts_df = updated_df.groupBy("customer_id").agg(
            count(when(col("is_delayed") == 1, True)).alias("total_delay_count"),
            count("*").alias("total_payment_count")
        )

        # Calculating delay ratio for customers
        cust_delay_ratios_df = cust_delay_total_counts_df.withColumn(
            "delay_ratio",
            (col("total_delay_count") / col("total_payment_count")) * 100
        )

        # Flaging customers where delay ration > 50 percentage
        cust_delay_flaging_df = cust_delay_ratios_df.withColumn(
            "is_over_50_percent_delayed",
            when(col("delay_ratio") > 50, "Yes").otherwise("No")
        )

        return cust_delay_flaging_df
    
    def customerUnPaidAmount(self, payments_df, invoices_df):
        # Joining both invoices with payments
        invoices_with_payments_df = invoices_df.join(
            payments_df, 
            (invoices_df["customer_id"] == payments_df["customer_id"]) &
            (invoices_df["invoice_id"] == payments_df["invoice_id"]) &
            (payments_df["payment_date"] > invoices_df["invoice_date"]), 
            "left"
        )

        # Filtering unpaid invoices
        cust_unpaid_invoices_df = invoices_with_payments_df.filter(col("payment_date").isNull())

        # Aggreagting total unpaid amount for customers
        customers_agg_unpaid_df = cust_unpaid_invoices_df.groupBy("customer_id").agg(
            sum("amount_due").alias("total_unpaid_amount")
        )

        return customers_agg_unpaid_df
    
# Instantiating the class    
pay_tract_inst = CustomerPaymentBehaviour()

# Reading the files as dataframes
payments_df = pay_tract_inst.read_data("/Volumes/db_prac/prac/raw/payments")
invoices_df = pay_tract_inst.read_data("/Volumes/db_prac/prac/invoices")

# Reading the methods
invoices_with_payments_df = pay_tract_inst.invoices_with_payments(payments_df, invoices_df)
cust_avg_delay_df = pay_tract_inst.avgPaymentDelay(payments_df, invoices_df)
cust_delay_flaging_df = pay_tract_inst.flagingCustomers(payments_df, invoices_df)
customers_agg_unpaid_df = pay_tract_inst.customerUnPaidAmount(payments_df, invoices_df)

# Showing dataframes
invoices_with_payments_df.show()
cust_avg_delay_df.show()
cust_delay_flaging_df.show()
customers_agg_unpaid_df.show()

In [0]:
"""
Ranking With Multi-Level Tie Breaks
Dataset:
players(player_id, tournament_id, score, play_time_seconds)

Tasks:
For each tournament:
Rank players:
Highest score → lower play_time → alphabetical player_id
Identify players who consistently rank top 3 across tournaments (at least 3 times).
"""

In [0]:
# Sample data
data = [
    (1, 'T1', 95, 360),
    (2, 'T1', 95, 400),
    (3, 'T1', 90, 300),
    (4, 'T1', 95, 360),
    (1, 'T2', 88, 500),
    (2, 'T2', 92, 450),
    (3, 'T2', 92, 420),
    (5, 'T2', 92, 420),
    (1, 'T3', 96, 350),
    (2, 'T3', 96, 355),
    (3, 'T3', 89, 400),
    (5, 'T3', 96, 350),
    (6, 'T3', 96, 340),
]

# Define schema
schema = StructType([
    StructField("player_id", IntegerType(), False),
    StructField("tournament_id", StringType(), False),
    StructField("score", IntegerType(), False),
    StructField("play_time_seconds", IntegerType(), False)
])

# Create DataFrame
df = spark.createDataFrame(data, schema=schema)

df.show()

In [0]:
# Rank players: Highest score → lower play_time → alphabetical player_id

# Window specifictaion for ranking players
window_spec = Window.partitionBy("tournament_id").orderBy(desc("score"), asc("play_time_seconds"), "player_id")

players_ranking_df = df.withColumn(
    "player_rank",
    dense_rank().over(window_spec)
)
players_ranking_df.show()

# Identify players who consistently rank top 3 across tournaments (at least 3 times).

# Checking the players rank in threshold
ranking_analysis_df = players_ranking_df.withColumn(
    "is_in_threshold",
    when(col("player_rank") <= 3, 1).otherwise(0)
)
ranking_analysis_df.show()

# Aggregating total consistency ranking for players
aggregated_df = ranking_analysis_df.groupBy("player_id").agg(
    sum("is_in_threshold").alias("total_consistences")
)

# Filtering players where total consistency >= 3 times
filtered_df = aggregated_df.filter(col("total_consistences") >= 2)
filtered_df.show()


In [0]:
"""
Product Exchange Pattern
Datasets:
orders(order_id, customer_id, product_id, order_date)
returns(return_id, order_id, return_date)
new_orders(order_id, customer_id, product_id, order_date) (replacement order)

Tasks:
Link return → new order by customer and return_date ± 3 days.
For each product:
Count how often it was exchanged instead of returned for refund.
Calculate exchange ratio = exchanges / total returns.
"""

In [0]:
orders_data = [
    (101, 1, 'P1', '2024-06-01'),
    (102, 2, 'P2', '2024-06-03'),
    (103, 3, 'P3', '2024-06-05'),
    (104, 1, 'P4', '2024-06-07'),
    (105, 4, 'P1', '2024-06-08'),
]

orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("product_id", StringType(), False),
    StructField("order_date", StringType(), False),
])

orders_df_raw = spark.createDataFrame(orders_data, schema=orders_schema)

# Convert to proper DateType
orders_df = orders_df_raw.withColumn("order_date", to_date("order_date", "yyyy-MM-dd"))
orders_df.show()

returns_data = [
    (201, 101, '2024-06-10'),
    (202, 102, '2024-06-12'),
    (203, 105, '2024-06-15'),
]

returns_schema = StructType([
    StructField("return_id", IntegerType(), False),
    StructField("order_id", IntegerType(), False),
    StructField("return_date", StringType(), False),
])

returns_df_raw = spark.createDataFrame(returns_data, schema=returns_schema)

# Convert to DateType
returns_df = returns_df_raw.withColumn("return_date", to_date("return_date", "yyyy-MM-dd"))
returns_df.show()

new_orders_data = [
    (301, 1, 'P1', '2024-06-11'),
    (302, 2, 'P2', '2024-06-18'),
    (303, 4, 'P1', '2024-06-16'),
]

new_orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("product_id", StringType(), False),
    StructField("order_date", StringType(), False),
])

new_orders_df_raw = spark.createDataFrame(new_orders_data, schema=new_orders_schema)

# Convert to DateType
new_orders_df = new_orders_df_raw.withColumn("order_date", to_date("order_date", "yyyy-MM-dd"))
new_orders_df.show()

In [0]:
orders_with_returns_df = orders_df.join(returns_df, on="order_id", how="inner")

# Joining orders_with_returns with new orders
orders_updated_df = orders_with_returns_df.alias("df1").join(
    new_orders_df.alias("df2"),
    (col("df1.customer_id") == col("df2.customer_id")) &
    (col("df1.product_id") == col("df2.product_id")) &
    (col("df2.order_date").between(col("df1.return_date"), date_add(col("return_date"), 3))),
    "left"
)
orders_updated_df.show()

In [0]:
 class productExchangePattern:
    def read_data(self, path):
        try:
            df = spark.read.format('csv').option("header", True).load(path)
            return df
        except Exception as err:
            print(f"Error occur while reading file {path} : {err}")
            return None
        
    def exchangeAnalysis(self, orders_df, returns_df, new_orders_df):
        # Joining orders with returns to get only returned orders
        orders_with_returns_df = orders_df.join(returns_df, on="order_id", how="inner")

        # Joining orders_with_returns with new orders
        orders_updated_df = orders_with_returns_df.alias("df1").join(
            new_orders_df.alias("df2"),
            (col("df1.customer_id") == col("df2.customer_id")) &
            (col("df1.product_id") == col("df2.product_id")) &
            (col("df2.order_date").between(col("df1.return_date"), date_add(col("return_date"), 3))),
            how="left"
        )

        # Selecting required columns
        orders_updated_df = orders_updated_df.select(
            col("df1.customer_id").alias("customer_id"),
            col("df1.product_id").alias("product_id"),
            col("df1.return_id").alias("return_id"),
            col("df2.product_id").alias("exc_product_id")
        )

        # Aggregating total exchanges and total returns for each product
        products_exc_return_counts_df = orders_updated_df.groupBy("product_id").agg(
            count("exc_product_id").alias("total_exchanges"),
            count("return_id").alias("total_returns")
        )    

        # Calculating Products exchange ratio
        products_exc_ratio_df = products_exc_return_counts_df.withColumn(
            "exchange_ratio",
            col("total_exchanges") / col("total_returns")
        ) 

        return orders_updated_df, products_exc_return_counts_df, products_exc_ratio_df
    
# Instantiating the class
exc_inst = productExchangePattern()

# Reading files as dataframes
orders_df = exc_inst.read_data("/Volumes/db_prac/prac/raw/orders")
returns_df = exc_inst.read_data("/Volumes/db_prac/prac/raw/returns")
new_orders_df = exc_inst.read_data("/Volumes/db_prac/prac/raw/new_orders")

# Reading files
orders_updated_df, products_exc_return_counts_df, products_exc_ratio_df = exc_inst.exchangeAnalysis(orders_df, returns_df, new_orders_df)

# Showing dataframes
orders_updated_df.show()
products_exc_return_counts_df.show(),
products_exc_ratio_df.show()

In [0]:
"""
Login Anomaly Detection with Geolocation
Dataset:
logins(user_id, login_time, location, ip_address)

Tasks:
For each user:
Sort logins chronologically.
Calculate time difference and distance between logins.
Flag:
Logins < 1 hour apart but > 1000 km distance.
Same IP used by 3+ users in 10 minutes.
"""

In [0]:
# Define schema with login_time as string

schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("login_time", StringType(), True),
    StructField("location", StringType(), True),
    StructField("ip_address", StringType(), True)
])

# Sample data
data = [
    ("U1", "2024-05-01 10:00:00", "28.6139,77.2090", "192.168.1.1"),
    ("U1", "2024-05-01 10:45:00", "19.0760,72.8777", "192.168.1.1"),
    ("U1", "2024-05-01 12:00:00", "28.6139,77.2090", "192.168.1.2"),

    ("U2", "2024-05-01 10:05:00", "13.0827,80.2707", "192.168.1.1"),
    ("U2", "2024-05-01 10:12:00", "13.0827,80.2707", "192.168.1.1"),

    ("U3", "2024-05-01 10:07:00", "12.9716,77.5946", "192.168.1.1"),

    ("U4", "2024-05-01 11:00:00", "40.7128,-74.0060", "203.0.113.1"),
    ("U4", "2024-05-01 11:30:00", "34.0522,-118.2437", "203.0.113.2")
]

# Create DataFrame
df_raw = spark.createDataFrame(data, schema)

df = df_raw.withColumn("login_time", to_timestamp("login_time", "yyyy-MM-dd HH:mm:ss"))
df.printSchema()
df.show(truncate=False)

In [0]:
# Window spec to sort the logins chronologically
window_spec = Window.partitionBy("user_id").orderBy("login_time")

users_lat_lon_df = df.withColumn(
    "lat",
    split("location", ",")[0].cast("double")
).withColumn(
    "lon",
    split("location", ",")[1].cast("double")
)

users_prev_det_df = users_lat_lon_df.withColumn(
    "prev_login",
    lag("login_time").over(window_spec)
).withColumn(
    "prev_lat",
    lag("lat").over(window_spec)
).withColumn(
    "prev_lon",
    lag("lon").over(window_spec)
)

users_logins_distances_df = users_prev_det_df.withColumn(
    "time_difference",
    (unix_timestamp(col("login_time")) - unix_timestamp(col("prev_login"))) / 3600
)
# Earth radius in km
R = 6371.0

users_distances_df = users_logins_distances_df.withColumn("dlat", radians(col("lat") - col("prev_lat"))) \
       .withColumn("dlon", radians(col("lon") - col("prev_lon"))) \
       .withColumn("a", sin(col("dlat") / 2) ** 2 +
                        cos(radians(col("lat"))) * cos(radians(col("prev_lat"))) *
                        sin(col("dlon") / 2) ** 2) \
       .withColumn("c", 2 * atan2(sqrt(col("a")), sqrt(1 - col("a")))) \
       .withColumn("distance_km", R * col("c"))

# Flaging users logins
flagging_logins_df = users_distances_df.withColumn(
    "is_threshold_aligned",
    when((col("time_diff") < 1) & (col("distance_km") > 1000), "Yes").otherwise("No")
)

# Step 6: Detect same IP used by 3+ users within 10 minutes
# Self join on IP and 10-minute login window
joined_df = df.alias("df1").join(
    df.alias("df2"),
    (col("df1.ip_address") == col("df2.ip_address")) &
    (col("df2.login_time").between(
        col("df1.login_time"), 
        col("df1.login_time") + expr("INTERVAL 10 MINUTES")
    )),
    "inner"
)

# Selecting required columns from dataframe
updated_df = joined_df.select(
    col("df1.ip_address").alias("ip_address"),
    col("df1.login_time").alias("login_time"),
    col("df2.user_id").alias("user_id")
)

# Aggregating total distinct users for each 10 minutes window
aggregated_df = updated_df.groupBy("ip_address", "login_time").agg(
    countDistinct("user_id").alias("total_users")
)

# Flagging ip address 
flagging_ip_df = aggregated_df.withColumn(
    "is_three_users",
    when(col("total_users") >= 3, "Yes").otherwise("No")
)


# Show flagged anomalies
print("🚨 Logins flagged due to distance-time anomaly:")
flagging_logins_df.select("user_id", "login_time", "prev_login", "time_diff", "distance_km", "is_threshold_aligned").show(truncate=False)

print("🚨 IPs flagged for 3+ users within 10 minutes:")
flagging_ip_df.show(truncate=False)

In [0]:
inbound_data = [
    ("p1", 100, "2023-07-14 10:00:00"),
    ("p2", 50, "2023-07-14 11:30:00"),
    ("p1", 30, "2023-07-14 12:00:00"),
    ("p3", 70, "2023-07-14 13:15:00")
]

inbound_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("units_in", IntegerType(), True),
    StructField("timestamp_str", StringType(), True)
])

inbound_df_raw = spark.createDataFrame(inbound_data, schema=inbound_schema)

inbound_df = inbound_df_raw.withColumn("timestamp", to_timestamp("timestamp_str"))
inbound_df = inbound_df.drop("timestamp_str")
inbound_df.write.format("parquet").save("/Volumes/db_prac/prac/raw/inbounds")

outbound_data = [
    ("p1", 40, "2023-07-14 10:30:00"),
    ("p2", 60, "2023-07-14 11:45:00"),
    ("p1", 20, "2023-07-14 13:00:00"),
    ("p3", 100, "2023-07-14 14:00:00")
]

outbound_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("units_out", IntegerType(), True),
    StructField("timestamp_str", StringType(), True)
])

outbound_df_raw = spark.createDataFrame(outbound_data, schema=outbound_schema)

outbound_df = outbound_df_raw.withColumn("timestamp", to_timestamp("timestamp_str"))
outbound_df = outbound_df.drop("timestamp_str")
outbound_df.write.format("parquet").save("/Volumes/db_prac/prac/raw/outbounds")


In [0]:
# Schema with order_date as string initially
orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("order_date", StringType(), True)
])

# Sample data (order_date as string)
orders_data = [
    (1, 101, 501, "2025-07-01 10:30:00"),
    (2, 101, 501, "2025-07-05 12:15:00"),
    (3, 101, 501, "2025-07-10 09:45:00"),
    (4, 102, 502, "2025-07-03 14:10:00"),
    (5, 102, 502, "2025-07-08 11:20:00"),
    (6, 103, 503, "2025-07-02 16:50:00"),
    (7, 103, 503, "2025-07-09 08:05:00"),
    (8, 101, 501, "2025-07-15 17:30:00"),
    (9, 104, 504, "2025-07-04 13:00:00"),
    (10, 104, 504, "2025-07-11 10:00:00")
]

orders_df = spark.createDataFrame(orders_data, schema=orders_schema)

# Convert string to TimestampType
orders_df = orders_df.withColumn("order_date", to_timestamp("order_date", "yyyy-MM-dd HH:mm:ss"))

orders_df.write.format("csv").option("header", True).save("/Volumes/db_prac/prac/raw/orders")

# Schema with return_date as string initially
returns_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("return_date", StringType(), True)
])

# Sample data (return_date as string)
returns_data = [
    (1, "2025-07-03 09:00:00"),
    (2, "2025-07-06 15:30:00"),
    (4, "2025-07-04 10:00:00"),
    (6, "2025-07-04 18:45:00"),
    (8, "2025-07-16 19:20:00")
]

returns_df = spark.createDataFrame(returns_data, schema=returns_schema)

# Convert string to TimestampType
returns_df = returns_df.withColumn("return_date", to_timestamp("return_date", "yyyy-MM-dd HH:mm:ss"))

returns_df.write.format("csv").option("header", True).save("/Volumes/db_prac/prac/raw/returns")


In [0]:
orders_df.show()

returns_df.show()

In [0]:
# Campaigns schema
campaigns_schema = StructType([
    StructField("campaign_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("start_date", StringType(), False),
    StructField("end_date", StringType(), False)
])

# Sales schema
sales_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("sale_date", StringType(), False),
    StructField("units_sold", IntegerType(), False),
    StructField("category", StringType(), False)
])

# Campaigns data with string dates
campaigns_data = [
    (1, 101, "2025-06-01", "2025-06-10"),
    (2, 102, "2025-06-05", "2025-06-15"),
    (3, 103, "2025-06-08", "2025-06-18")
]

campaigns_df = spark.createDataFrame(campaigns_data, schema=campaigns_schema)

# Sales data with string dates
sales_data = [
    (101, "2025-05-29", 10, "Electronics"),
    (101, "2025-06-03", 25, "Electronics"),
    (102, "2025-06-07", 20, "Electronics"),
    (102, "2025-06-11", 15, "Electronics"),
    (103, "2025-06-02", 12, "Home"),
    (104, "2025-05-30", 18, "Electronics"),
    (104, "2025-06-04", 30, "Electronics"),
    (105, "2025-06-06", 22, "Electronics"),
    (106, "2025-06-09", 35, "Home"),
    (106, "2025-06-12", 40, "Home"),
]

sales_df = spark.createDataFrame(sales_data, schema=sales_schema)

campaigns_df = campaigns_df \
    .withColumn("start_date", to_date(col("start_date"), "yyyy-MM-dd")) \
    .withColumn("end_date", to_date(col("end_date"), "yyyy-MM-dd"))

sales_df = sales_df \
    .withColumn("sale_date", to_date(col("sale_date"), "yyyy-MM-dd"))

campaigns_df.show()
sales_df.show()

In [0]:
class SKUSalesImpact:
    def read_data(self, path):
        try:
            df = spark.read.format('csv').option("header", True).load(path)
            return df
        except Exception as err:
            print(f"Error occur while reading file {path} : {err}")
            return None
        

    def campaignProductSales(self, campaigns_df, sales_df):
        # Joining campaign products with sales with in window
        products_sales_df = campaigns_df.alias("cam").join(
            sales_df.alias("sales"),
            (col("cam.product_id") == col("sales.product_id")) &
            (col("sales.sale_date").between(col("cam.start_date"), col("cam.end_date"))),
            "inner"
        )

        # Selecting required columns
        products_sales_df = products_sales_df.select(
            col("cam.campaign_id").alias("campaign_id"),
            col("cam.product_id").alias("product_id"),
            col("sales.units_sold").alias("units_sold"),
            col("sales.category").alias("category")
        )

        # Aggregating units sold for campaign products
        products_during_window_sales_df = products_sales_df.groupBy("product_id", "campaign_id").agg(
            sum("units_sold").alias("during_sales")
        )


        # joining products with sales to get category
        products_category_df = campaigns_df.alias("cam").join(
            sales_df.alias("sales"),
            col("cam.product_id") == col("sales.product_id"),
            "inner"
        ).select("cam.campaign_id", "cam.product_id", "sales.category", "cam.start_date", "cam.end_date").distinct()

        # Joining with sales to get related products from categories
        related_products_df = products_category_df.alias("pc").join(
            sales_df.alias("sales"),
            (col("pc.category") == col("sales.category")) &
            (col("pc.product_id") != col("sales.product_id")) &
            (col("sales.sale_date").between(col("pc.start_date"), col("pc.end_date"))),
            "left"
        )

        # selecting required columns 
        related_products_df = related_products_df.select(
            col("sales.product_id").alias("product_id"),
            col("sales.units_sold").alias("units_sold"),
            col("sales.category").alias("category")
        )

        # Aggregating sales for related products
        related_products_total_sales_df = related_products_df.groupBy("product_id").agg(
            sum("units_sold").alias("rel_product_sales")
        )

        return products_during_window_sales_df, related_products_total_sales_df


    def salesComparisions(self, campaigns_df, sales_df):
        # Calling the products sales function
        products_during_window_sales_df, _ = self.campaignProductSales(campaigns_df, sales_df)

        # Creating before window columns
        campaigns_df = campaigns_df.withColumn(
            "duration",
            datediff(col("end_date"), col("start_date"))+1
        ).withColumn(
            "before_window_start",
            date_sub(col("start_date"), col("duration"))
        ).withColumn(
            "before_window_end",
            date_sub(col("start_date"), 1)
        )

        # Joining products sales with in before window
        products_before_window_data_df = campaigns_df.alias("cam").join(
            sales_df.alias("sales"),
            (col("cam.product_id") == col("sales.product_id")) &
            (col("sales.sale_date").between(col("cam.before_window_start"), col("cam.before_window_end"))),
            "left"
        ).fillna(0)

        # Aggregating total sales
        products_before_window_sales_df = products_before_window_data_df.groupBy("cam.product_id").agg(
            sum("sales.units_sold").alias("before_sales")
        )

        # Joining products during and before window data
        products_during_before_sales_df = products_during_window_sales_df.alias("pdw").join(
            products_before_window_sales_df.alias("pbw"),
            (col("pdw.product_id") == col("pbw.product_id")),
            "left"
        )

        # Selecting required columns
        products_during_before_sales_df = products_during_before_sales_df.select(
            col("pdw.campaign_id").alias("campaign_id"),
            col("pdw.product_id").alias("product_id"),
            col("pdw.during_sales").alias("during_sales"),
            col("pbw.before_sales").alias("before_sales")
        )

        # Checking the difference between during and before windows sales
        product_sales_diff_df = products_during_before_sales_df.withColumn(
            "comparision_diff",
            col("during_sales") - col("before_sales")
        )

        return product_sales_diff_df
    

    def campaignFlagging(self, campaigns_df, sales_df):
        # Reading function
        product_sales_diff_df = self.salesComparisions(campaigns_df, sales_df)

        # Creating sku change column
        products_sku_changes_df = product_sales_diff_df.withColumn(
            "sku_change",
            when(col("before_sales") > 0, ((col("during_sales") - col("before_sales")) / col("before_sales")) * 100).otherwise(0)
        )

        # Flagging campaigns
        campaign_flaggings_df = products_sku_changes_df.withColumn(
            "is_greater_20",
            when(col("sku_change") >= 20, "Yes").otherwise("No")
        )

        return campaign_flaggings_df
    

# Instantiating class
sku_inst = SKUSalesImpact()    

# Reading file as dataframe
campaigns_df = sku_inst.read_data("/Volumes/db_prac/prac/campaigns")
sales_df = sku_inst.read_data("/Volumes/db_prac/prac/sales")

# Reading methods
products_during_window_sales_df, related_products_total_sales_df = sku_inst.campaignProductSales(campaigns_df, sales_df)        
product_sales_diff_df = sku_inst.salesComparisions(campaigns_df, sales_df)
campaign_flaggings_df = sku_inst.campaignFlagging(campaigns_df, sales_df)
        

# Showing dataframes
products_during_window_sales_df.show()
related_products_total_sales_df.show()
product_sales_diff_df.show()
campaign_flaggings_df.show()        

In [0]:
# Interactions schema
interactions_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("source", StringType(), True),
    StructField("interaction_time", TimestampType(), True)
])

# Purchases schema
purchases_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("purchase_time", TimestampType(), True)
])

# Create DataFrames
interactions_df = spark.createDataFrame([
    (1, "Google", "2025-07-10 10:00:00"),
    (1, "Facebook", "2025-07-12 09:30:00"),
    (2, "LinkedIn", "2025-07-11 14:00:00"),
    (2, "Google", "2025-07-15 16:30:00"),
    (3, "Instagram", "2025-07-13 18:00:00"),
    (3, "Google", "2025-07-18 12:00:00")
], schema=interactions_schema)

purchases_df = spark.createDataFrame([
    (1, "2025-07-13 11:00:00"),
    (2, "2025-07-16 15:00:00"),
    (3, "2025-07-19 10:00:00")
], schema=purchases_schema)

interactions_df.show()
purchases_df.show()

In [0]:
# Join most recent source interaction before purchase.
window_spec = Window.partitionBy("user_id").orderBy("purchase_time")
purchases_df = purchases_df.withColumn(
    "prev_purchase_time",
    lag("purchase_time").over(window_spec)
)

purchases_df = purchases_df.withColumn(
    "prev_purchase_time",
    coalesce(col("prev_purchase_time"), lit("1900-01-01 00:00:00").cast("timestamp"))
)

# Joining purchases with there interactions
purchases_with_interactions_df = purchases_df.alias("purc").join(
    interactions_df.alias("inter"),
    (col("purc.user_id") == col("inter.user_id")) &
    (col("inter.interaction_time").between(col("purc.prev_purchase_time"), col("purc.purchase_time"))),
    "inner"
)

purchases_with_interactions_df.show()

# If multiple, assign weight by recency (e.g., exp(-days_diff)).

# Checking the time difference between purchases and interactions
purchases_time_diff_df = purchases_with_interactions_df.withColumn(
    "time_diff",
    (unix_timestamp("purchase_time") - unix_timestamp("interaction_time")) / 3600
)

# Calculating weight for interactions
interaction_weights_df = purchases_time_diff_df.withColumn(
    "weight",
    exp(-col("time_diff"))
)

interaction_weights_df.show()

# Ranking interactions weight for purchases
window_rnk = Window.partitionBy("user_id", "purchase_time").orderBy(desc("weight"))

interactions_rankings_df = interaction_weights_df.withColumn(
    "weight_rnk",
    dense_rank().over(window_rnk)
)
interactions_rankings_df.show()

# Aggregate weighted conversions by source.

# Aggregating total weighted conversions for each source
sources_weights_agg_df = interaction_weights_df.groupBy("source").agg(
    sum("weight").alias("total_weighted_conv")
)

# Ranking sources based on conversions
window_rnk2 = Window.orderBy(desc("total_weighted_conv"))

ranking_sources_df = sources_weights_agg_df.withColumn(
    "source_rnk",
    dense_rank().over(window_rnk2)
)

print("=== Interactions Ranked by Weight ===")
interactions_rankings_df.show(truncate=False)

print("=== Sources Ranked by Weighted Conversions ===")
ranking_sources_df.show(truncate=False)


In [0]:
"""
Exam Time Slot Clash Detector
Dataset:
exam_schedule(student_id, subject, exam_start, exam_end)

Tasks:
For each student:
Detect overlapping exams (start1 < end2 AND start2 < end1).
Count number of clashes.
List subjects involved in each clash.
"""

In [0]:
schema = StructType([
    StructField("student_id", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("exam_start", StringType(), True),
    StructField("exam_end", StringType(), True),
])

data = [
    ("S001", "Math",    "2025-07-29 09:00:00", "2025-07-29 11:00:00"),
    ("S001", "Physics", "2025-07-29 10:30:00", "2025-07-29 12:00:00"),
    ("S001", "English", "2025-07-29 13:00:00", "2025-07-29 15:00:00"),
    ("S002", "Chem",    "2025-07-29 09:00:00", "2025-07-29 11:00:00"),
    ("S002", "Bio",     "2025-07-29 11:30:00", "2025-07-29 13:00:00"),
    ("S002", "Geo",     "2025-07-29 11:00:00", "2025-07-29 12:30:00"),
    ("S003", "History", "2025-07-29 08:00:00", "2025-07-29 10:00:00"),
    ("S003", "Civics",  "2025-07-29 10:00:00", "2025-07-29 12:00:00")
]

# Create DataFrame with string timestamps
exam_schedule_df_raw = spark.createDataFrame(data, schema=schema)

exam_schedule_df = exam_schedule_df_raw.withColumn("exam_start", to_timestamp("exam_start")) \
                                       .withColumn("exam_end", to_timestamp("exam_end"))

exam_schedule_df.show(truncate=False)

In [0]:
# For each student Detect overlapping exams (start1 < end2 AND start2 < end1).

# Making self join over exam_schedules to detect overlaps 
exams_overlaps_df = exam_schedule_df.alias("e1").join(
    exam_schedule_df.alias("e2"),
    (col("e1.student_id") == col("e2.student_id")) &
    (col("e1.exam_start") < col("e2.exam_end")) &
    (col("e2.exam_start") < col("e1.exam_end")) &
    (col("e1.subject") != col("e2.subject")) &
    (col("e1.subject") < col("e2.subject")),
    "inner"
)

# Selecting required columns
exams_overlaps_df = exams_overlaps_df.select(
    col("e1.student_id").alias("student_id"),
    col("e1.subject").alias("subject"),
    col("e1.exam_start").alias("clash_start_time"),
    col("e2.subject").alias("conflictingsubject")
)

# Aggregating number of clashes for each student
student_clashes_count_df = exams_overlaps_df.groupBy("student_id").agg(
    countDistinct("clash_start_time").alias("total_clashes")
)

# List subjects involved in each clash.

# Creating clash pair column by concatinating subject and conflictsubject

exams_overlaps_df = exams_overlaps_df.withColumn(
    "clash_pair",
    concat_ws("_", col("subject"), col("conflictingsubject"))
)

student_clash_sub_collect_df = exams_overlaps_df.groupBy("student_id", "clash_start_time").agg(
    collect_list("clash_pair").alias("total_subjects")
)



In [0]:
"""
Fraud Pattern Detection in Transactions
Datasets:
transactions(user_id, txn_id, txn_amount, txn_time)
devices(user_id, device_id, location)

Tasks:
Find users who:
Made 2+ transactions within 10 minutes from different devices.
Totaled over ₹50,000 in 30 mins.
Flag those as suspicious.
"""

In [0]:
transactions_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("txn_id", StringType(), True),
    StructField("txn_amount", DoubleType(), True),
    StructField("txn_time", StringType(), True),
    StructField("device_id", StringType(), True)
])

transactions_data = [
    ("u1", "t1", 30000, "2023-01-01 10:00:00", "d1"),
    ("u1", "t2", 25000, "2023-01-01 10:05:00", "d2"),
    ("u1", "t3", 15000, "2023-01-01 10:35:00", "d1"),
    ("u2", "t4", 10000, "2023-01-01 09:00:00", "d3"),
    ("u2", "t5", 20000, "2023-01-01 09:20:00", "d3"),
    ("u2", "t6", 30000, "2023-01-01 09:25:00", "d3"),
    ("u3", "t7", 1000,  "2023-01-01 11:00:00", "d4"),
]

transactions_df = spark.createDataFrame(data=transactions_data, schema=transactions_schema)

transactions_df = transactions_df.withColumn("txn_time", to_timestamp("txn_time", "yyyy-MM-dd HH:mm:ss"))
transactions_df.show()

devices_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("location", StringType(), True)
])

devices_data = [
    ("u1", "d1", "Hyderabad"),
    ("u1", "d2", "Mumbai"),
    ("u2", "d3", "Delhi"),
    ("u2", "d3", "Delhi"),
    ("u3", "d4", "Bangalore"),
]

devices_df = spark.createDataFrame(data=devices_data, schema=devices_schema)
devices_df.show()

In [0]:
# Find users who Made 2+ transactions within 10 minutes from different devices.

# Converting txn_time to unix_timestamp to get long string as seconds
transactions_df = transactions_df.withColumn("txn_ts", unix_timestamp(col("txn_time")))

# Window specification for 10 minutes sliding window for users transactions
window_spec = Window.partitionBy("user_id").orderBy("txn_ts").rangeBetween(-600, 0)

users_10_min_agg_df = transactions_df.withColumn(
    "total_transactions",
    count("txn_id").over(window_spec)
)

# Checking devices over user transactions
users_10_min_agg_df = users_10_min_agg_df.withColumn(
    "prev_device_id",
    lag("device_id").over(window_spec)
)

# Checking device conversions
users_10_min_agg_df = users_10_min_agg_df.withColumn(
    "device_change_flag",
    when((col("prev_device_id").isNull()) | (col("device_id") != col("prev_device_id")), 1).otherwise(0)
)

# Aggregating total devices for users windows
users_total_transactions_devices_df = users_10_min_agg_df.withColumn(
    "total_devices",
    sum("devdevice_change_flag").over(window_spec)
)

# Flagging suspecious users
flagging_df = users_total_transactions_devices_df.withColumn(
    "is_suspecious",
    when((col("total_transactions") >= 2) & (col("total_devices") >= 2), "Yes").otherwise("No")
)

# Filtering only suspecious users from flagging df
suspecious_df = flagging_df.filter(col("is_suspecious") == "Yes")

suspecious_df.show()

# Finding users Totaled over ₹50,000 in 30 mins.

# Window_specification for 30 minutes window 
window_spectwo = Window.partitionBy("user_id").orderBy("txn_ts").rangeBetween(-1800, 0)

# Aggregating total amount for users 30 minutes windows
users_30_minutes_df = transactions_df.withColumn(
    "total_amount",
    sum("txn_amount").over(window_spectwo)
)

# Flagging users who has 50000 in 30 minutes window
flagging_users_df = users_30_minutes_df.withColumn(
    "is_50000",
    when(col("total_amount") > 50000, "Yes").otherwise("No")
)

# Filtering users
final_50000_users_df = flagging_users_df.filter(col("is_50000") == "Yes")


# Union both user sets and get distinct user_ids
final_df = (
    suspecious_df.select("user_id")
    .union(final_50000_users_df.select("user_id"))
    .distinct()
)
final_df.show()

In [0]:
"""
Shopping Cart Abandonment Analyzer
Datasets:
cart_events(user_id, product_id, event_time, action)
(action: add, remove, purchase)

Tasks:
For each user-product:
Identify abandoned carts (added but not purchased within 1 hour).
Count total abandoned carts per user.
Rank users by abandonment rate (use window functions).
"""

In [0]:
# Schema with event_time as string initially
schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("event_time", StringType(), True),  # Note: initially a string
    StructField("action", StringType(), True)
])

# Sample data
data = [
    ("u1", "p1", "2024-07-01 10:00:00", "add"),
    ("u1", "p1", "2024-07-01 10:30:00", "remove"),
    ("u1", "p2", "2024-07-01 11:00:00", "add"),
    ("u1", "p2", "2024-07-01 12:05:00", "purchase"),
    ("u2", "p1", "2024-07-01 09:45:00", "add"),
    ("u2", "p1", "2024-07-01 11:00:00", "purchase"),
    ("u2", "p2", "2024-07-01 10:00:00", "add"),
    ("u3", "p3", "2024-07-01 10:00:00", "add"),
    ("u3", "p3", "2024-07-01 11:01:00", "purchase"),
    ("u3", "p4", "2024-07-01 11:15:00", "add"),
    ("u3", "p4", "2024-07-01 12:30:00", "remove"),
]

# Create DataFrame
cart_events_df = spark.createDataFrame(data, schema)

# Convert event_time from string to timestamp
cart_events_df = cart_events_df.withColumn("event_time", to_timestamp("event_time"))

# Show sample
cart_events_df.show(truncate=False)

In [0]:
class CartAbandonmentAnalyzer:
    def read_data(self, path):
        try:
            df = spark.read.format('csv').option("header", True).load(path)
            return df
        except Exception as err:
            print(f"Error occur while reading file {path} : {err}")
            return None
        
    def userProductsAbandonment(self, cart_events_df):
        # Filtering add actions from dataframe
        filtered_df = cart_events_df.filter(col("action") == "add")

        # Joining filtered dataframe with cart_events dataframe
        joined_df = filtered_df.alias("fil").join(
            cart_events_df.alias("cart"),
            (col("fil.user_id") == col("cart.user_id")) &
            (col("fil.product_id") == col("cart.product_id")) &
            (col("cart.action") != "add") &
            (col("cart.event_time").between(col("fil.event_time"), expr("fil.event_time + interval 1 hour"))),
            "left"
        )    

        # Selecting required columns
        joined_df = joined_df.select(
            col("fil.user_id").alias("user_id"),
            col("fil.product_id").alias("product_id"),
            col("fil.event_time").alias("added_time"),
            col("cart.action").alias("next_action")
        )

        # Filtering products which dont have purchases
        products_with_aband_carts_df = joined_df.filter(col("next_action").isNull())

        products_with_aband_carts_df.show()

        # Aggregating total abandonment products for each user
        user_total_aban_products_df = products_with_aband_carts_df.groupBy("user_id").agg(
            countDistinct("product_id").alias("total_aban_products")
        )


        # Aggregating total added products for each user
        user_total_products_df = filtered_df.groupBy("user_id").agg(
            countDistinct("product_id").alias("total_products")
        )

        # Joining user total added and abandoned products count
        user_with_added_aban_products_df = user_total_products_df.join(user_total_aban_products_df, on="user_id",how="left").fillna(0)

        users_aban_rate_df = user_with_added_aban_products_df.withColumn(
            "abandonment_rate",
            col("total_aban_products") / col("total_products")
        )

        # Window spec for ranking users based on abandonment rate
        window_rnk = Window.orderBy(desc("abandonment_rate"))

        users_ranking_df = users_aban_rate_df.withColumn(
            "rnk",
            dense_rank().over(window_rnk)
        )

        return products_with_aband_carts_df, user_total_aban_products_df, user_total_products_df, users_ranking_df
    
# Instantiate the class 
cart_inst = CartAbandonmentAnalyzer()

# Reading file from path
cart_events_df = cart_inst.read_data("/Volumes/db_prac/prac/raw/carts")

# Reading file 
products_with_aband_carts_df, user_total_aban_products_df, user_total_products_df, users_ranking_df = cart_inst.userProductsAbandonment(cart_events_df)

# Showing dataframes
products_with_aband_carts_df.show()
user_total_aban_products_df.show()
user_total_products_df.show()
users_ranking_df.show()

In [0]:
"""
Video Watch Completion Analysis
Dataset:
video_plays(user_id, video_id, play_time, watch_duration, video_length)

Tasks:
For each user-video:
Calculate completion %.
Rank top videos by avg completion.
Find users who watched 3+ videos >90% to completion
"""

In [0]:
schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("video_id", StringType(), True),
    StructField("play_time", StringType(), True),  # string for now
    StructField("watch_duration", IntegerType(), True),
    StructField("video_length", IntegerType(), True)
])

data = [
    ("u1", "v1", "2024-06-01 10:00:00", 280, 300),
    ("u1", "v2", "2024-06-02 11:00:00", 100, 300),
    ("u1", "v3", "2024-06-03 12:00:00", 275, 290),
    ("u2", "v1", "2024-06-01 09:00:00", 290, 300),
    ("u2", "v3", "2024-06-02 14:00:00", 150, 290),
    ("u3", "v2", "2024-06-01 13:00:00", 300, 300),
    ("u3", "v3", "2024-06-02 15:00:00", 270, 290),
    ("u3", "v4", "2024-06-03 15:00:00", 50, 300),
    ("u4", "v2", "2024-06-01 16:00:00", 260, 300),
    ("u4", "v4", "2024-06-02 17:00:00", 290, 300),
]

video_plays_df = spark.createDataFrame(data, schema)

video_plays_df = video_plays_df.withColumn("play_time", to_timestamp("play_time", "yyyy-MM-dd HH:mm:ss"))

video_plays_df.show(truncate=False)

In [0]:
# Calculate completion %.
video_plays_df = video_plays_df.withColumn(
    "completion_perc",
    round(col("watch_duration") / col("video_length") * 100, 0)
)

video_plays_df.show()

# Rank top videos by avg completion.
# Aggregating average completion for videos
vd_avg_comppletion_df = video_plays_df.groupBy("video_id").agg(
    avg("completion_perc").alias("avg_completion")
)

# Window spec for ranking videos
window_rnk = Window.orderBy(desc("avg_completion"))

ranking_videos_df = vd_avg_comppletion_df.withColumn(
    "rnk",
    dense_rank().over(window_rnk)
)

ranking_videos_df.show()

# Find users who watched 3+ videos >90% to completion.
threshold_df = video_plays_df.withColumn(
    "watch_90Plus",
    when(col("completion_perc") > 90.0, 1).otherwise(0)
)

# Aggregating both total videos and 90 plus for users
users_agg_df = threshold_df.groupBy("user_id").agg(
    count("video_id").alias("total_videos"),
    sum("watch_90Plus").alias("total_90Plus")
)

# Filtering users who met condition
filtered_df = users_agg_df.filter((col("total_videos") >= 3) & (col("total_90Plus") >= 3))

filtered_df.show()

In [0]:
data = [
    ("T1", "U1", "A1", "2025-07-01 10:00:00", "2025-07-01 12:30:00", "resolved"),
    ("T2", "U2", "A2", "2025-07-02 09:15:00", "2025-07-02 10:00:00", "resolved"),
    ("T3", "U3", "A1", "2025-07-03 11:00:00", "2025-07-03 14:00:00", "resolved"),
    ("T4", "U4", "A3", "2025-07-04 15:00:00", "2025-07-04 16:00:00", "resolved"),
    ("T5", "U2", "A2", "2025-07-05 08:00:00", "2025-07-05 09:30:00", "resolved"),
    ("T6", "U1", "A3", "2025-07-06 10:30:00", None,                 "open"),     # Unresolved
    ("T7", "U5", "A1", "2025-07-08 10:00:00", "2025-07-08 13:30:00", "resolved"),
    ("T8", "U6", "A2", "2025-07-09 09:00:00", "2025-07-09 09:45:00", "resolved"),
    ("T9", "U3", "A1", "2025-07-09 12:00:00", "2025-07-09 12:10:00", "resolved")
]

schema = StructType([
    StructField("ticket_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("agent_id", StringType(), True),
    StructField("created_time", StringType(), True),
    StructField("resolved_time", StringType(), True),
    StructField("status", StringType(), True)
])

tickets_df = spark.createDataFrame(data, schema)

# Convert string columns to timestamps
tickets_df = tickets_df.withColumn("created_time", to_timestamp("created_time")) \
                       .withColumn("resolved_time", to_timestamp("resolved_time"))

tickets_df.show(truncate=False)

tickets_df.write.format("parquet").mode("overwrite").save("/Volumes/db_prac/prac/raw/tickets")

In [0]:
# Define schema explicitly
schema = StructType([
    StructField("product_id", StringType(), nullable=False),
    StructField("sale_date", StringType(), nullable=False),  # string for now
    StructField("units_sold", IntegerType(), nullable=False)
])

# Sample data
data = [
    ("P1", "2025-06-01", 100),
    ("P1", "2025-06-08", 150),
    ("P1", "2025-06-15", 180),
    ("P1", "2025-06-22", 220),
    
    ("P2", "2025-06-01", 90),
    ("P2", "2025-06-08", 95),
    ("P2", "2025-06-15", 120),
    ("P2", "2025-06-22", 200),
    
    ("P3", "2025-06-01", 300),
    ("P3", "2025-06-08", 310),
    ("P3", "2025-06-15", 320),
    ("P3", "2025-06-22", 330),
    
    ("P4", "2025-06-01", 50),
    ("P4", "2025-06-08", 100),
    ("P4", "2025-06-15", 90),
    ("P4", "2025-06-22", 120),
    
    ("P5", "2025-06-01", 400),
    ("P5", "2025-06-08", 420),
    ("P5", "2025-06-15", 440),
    ("P5", "2025-06-22", 480),
    
    ("P6", "2025-06-01", 70),
    ("P6", "2025-06-08", 75),
    ("P6", "2025-06-15", 100),
    ("P6", "2025-06-22", 150),
]

# Create DataFrame
df = spark.createDataFrame(data, schema=schema)

df.show()

In [0]:
sales_schema = StructType([
    StructField("sale_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("sale_date", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("price", IntegerType(), False)
])

sales_data = [
    (1, 101, 1001, "2024-01-15", 10, 200),
    (2, 101, 1002, "2024-01-20", 5, 100),
    (3, 101, 1003, "2024-02-10", 8, 160),
    (4, 101, 1004, "2024-03-05", 15, 300),
    (5, 101, 1005, "2024-04-07", 12, 240),
    (6, 102, 1006, "2024-01-25", 20, 400),
    (7, 102, 1007, "2024-02-15", 18, 360),
    (8, 102, 1008, "2024-03-18", 25, 500),
    (9, 102, 1009, "2024-04-22", 22, 440),
    (10, 103, 1010, "2024-02-28", 30, 600),
    (11, 103, 1011, "2024-03-12", 28, 560),
    (12, 103, 1012, "2024-04-15", 35, 700),
]

sales_df = spark.createDataFrame(sales_data, schema=sales_schema)
sales_df = sales_df.withColumn("sale_date", to_date("sale_date"))

# -----------------------------
# product_targets dataset
# -----------------------------
product_targets_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("month", StringType(), False),
    StructField("target_quantity", IntegerType(), False)
])

product_targets_data = [
    (101, "2024-01", 12),
    (101, "2024-02", 10),
    (101, "2024-03", 20),
    (101, "2024-04", 15),
    (102, "2024-01", 18),
    (102, "2024-02", 15),
    (102, "2024-03", 22),
    (102, "2024-04", 20),
    (103, "2024-02", 25),
    (103, "2024-03", 28),
    (103, "2024-04", 30),
]

product_targets_df = spark.createDataFrame(product_targets_data, schema=product_targets_schema)

# Just to check
sales_df.show()
product_targets_df.show()

In [0]:
sales_df.write.mode("overwrite").format("parquet").save("/Volumes/db_prac/prac/raw/prod_sales")

product_targets_df.write.mode("overwrite").format("parquet").save("/Volumes/db_prac/prac/raw/product_targets")

In [0]:
"""
Assignment 2 – Active Customer Tracking
You have:
transactions: (txn_id, customer_id, txn_date, amount)
customers: (customer_id, join_date, city)

Task:
For each customer, find the date of first transaction and date of last transaction.

Mark a customer as "Active" if they have transactions in the last 60 days from today.

For each city, find the top 3 customers by total amount spent in the last year (use window rank).
"""

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import to_date

spark = SparkSession.builder.appName("ActiveCustomerTracking").getOrCreate()

# ---------------- Customers ----------------
customers_data = [
    (1, "2023-05-10", "New York"),
    (2, "2023-06-15", "Los Angeles"),
    (3, "2023-07-20", "Chicago"),
    (4, "2023-08-05", "New York"),
    (5, "2023-09-01", "Los Angeles"),
]

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("join_date", StringType(), True),  # initially string
    StructField("city", StringType(), True)
])

customers_df = spark.createDataFrame(customers_data, schema=customers_schema)

# ---------------- Transactions ----------------
transactions_data = [
    (101, 1, "2024-08-01", 200.0),
    (102, 1, "2024-12-15", 150.0),
    (103, 2, "2025-01-10", 500.0),
    (104, 2, "2025-02-05", 250.0),
    (105, 3, "2024-06-20", 300.0),
    (106, 3, "2025-01-25", 400.0),
    (107, 4, "2024-11-05", 100.0),
    (108, 4, "2025-02-15", 700.0),
    (109, 5, "2024-09-10", 350.0),
    (110, 5, "2025-02-10", 600.0),
]

transactions_schema = StructType([
    StructField("txn_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("txn_date", StringType(), True),   # initially string
    StructField("amount", DoubleType(), True)
])

transactions_df = spark.createDataFrame(transactions_data, schema=transactions_schema)

customers_df = customers_df.withColumn("join_date", to_date("join_date", "yyyy-MM-dd"))
transactions_df = transactions_df.withColumn("txn_date", to_date("txn_date", "yyyy-MM-dd"))

customers_df.show()
transactions_df.show()

In [0]:
# For each customer, find the date of first transaction and date of last transaction.

# Joining both customers and transactions 
customers_transactions_df = customers_df.join(transactions_df, on="customer_id", how="inner")

# Finding customers first and last transaction dates
cust_first_last_transacions_df = customers_transactions_df.groupBy("customer_id").agg(
    min("txn_date").alias("first_transaction_date"),
    max("txn_date").alias("last_transaction_date")
) 

# Flagging customers as active and inactive based on transactions history
customers_flagging_df = cust_first_last_transacions_df.withColumn(
    "flag",
    when(col("last_transaction_date") > date_sub(current_date(), 60), "Active").otherwise("InActive")
)

# For each city, find the top 3 customers by total amount spent in the last year (use window rank).

# Filtering last year data only from transactions
filtered_data_df = customers_transactions_df.filter(col("txn_date") > add_months(current_date(), -12))
# Aggregating total amount for customers per city 
city_totalAmounts_df = filtered_data_df.groupBy("customer_id", "city").agg(
    sum("amount").alias("total_amount")
)

# Window spec for ranking customers per city
window_spec = Window.partitionBy("city").orderBy(desc("total_amount"))

customers_ranking_df = city_totalAmounts_df.withColumn(
    "cust_rank",
    dense_rank().over(window_spec)
)

# Filtering top 3 customers per city
top3_customers_df = customers_ranking_df.filter(col("cust_rank") <= 3)
top3_customers_df.show()

In [0]:

# Employees Data
employees_data = [
    (1, "Alice", 101, "2020-01-15"),
    (2, "Bob", 101, "2019-03-20"),
    (3, "Charlie", 102, "2021-07-01"),
    (4, "David", 103, "2018-11-11"),
    (5, "Eve", 102, "2022-02-10")
]

employees_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("dept_id", IntegerType(), True),
    StructField("hire_date", StringType(), True)   # keeping as string first
])

employees_df = spark.createDataFrame(employees_data, employees_schema)

# Convert hire_date string -> actual DateType
employees_df = employees_df.withColumn("hire_date", to_date("hire_date", "yyyy-MM-dd"))

# Departments Data
departments_data = [
    (101, "HR"),
    (102, "Finance"),
    (103, "IT")
]

departments_schema = StructType([
    StructField("dept_id", IntegerType(), True),
    StructField("dept_name", StringType(), True)
])

departments_df = spark.createDataFrame(departments_data, departments_schema)

# Salaries Data
salaries_data = [
    (1, "2024-01", 5000),
    (1, "2024-02", 5200),
    (2, "2024-01", 4800),
    (2, "2024-02", 5000),
    (3, "2024-01", 7000),
    (3, "2024-02", 7100),
    (4, "2024-01", 6500),
    (4, "2024-02", 6400),
    (5, "2024-01", 4000),
    (5, "2024-02", 4200)
]
salaries_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("salary_month", StringType(), True),   # keep as string first
    StructField("salary_amount", DoubleType(), True)
])

salaries_df = spark.createDataFrame(salaries_data, salaries_schema)

# Convert salary_month string -> actual DateType
# salaries_df = salaries_df.withColumn("salary_month", to_date("salary_month", "yyyy-MM"))

# Show sample data
print("Employees:")
employees_df.show()

print("Departments:")
departments_df.show()

print("Salaries:")
salaries_df.show()


In [0]:
# Joining dataframes 
joined_df = employees_df.join(departments_df, on="dept_id", how="inner")

cust_dept_sal_df = joined_df.join(salaries_df, on="emp_id", how="inner")
cust_dept_sal_df.show()

# Departments total salaries per month
dept_total_sal_df = cust_dept_sal_df.groupBy("dept_id", "salary_month").agg(
    sum("salary_amount").alias("total_salary")
)
dept_total_sal_df.show()

# Departments average salary on that month
dept_avg_sal_df = cust_dept_sal_df.groupBy("dept_id", "salary_month").agg(
    avg("salary_amount").alias("dept_avg_salary")
)
dept_avg_sal_df.show()

# Joining with employees data and departments average salary
emp_sal_dept_avg_df = cust_dept_sal_df.join(
    dept_avg_sal_df,
    ["dept_id", "salary_month"],
    "inner"
)
emp_sal_dept_avg_df.show()

# Calculating deviation for employees
updated_df = emp_sal_dept_avg_df.withColumn(
    "deviation",
    col("salary_amount") - col("dept_avg_salary")
)

filtered_df = updated_df.filter(col("deviation") > (0.2 * col("dept_avg_salary")))
filtered_df.show()

In [0]:
"""
Assignment 5 – Multi-Level Ranking
You have:
orders: (order_id, customer_id, order_date, order_amount)
regions: (customer_id, region)

Task:
For each region, find the top 5 customers by total order amount.
For each customer in the top 5, rank their orders from highest to lowest by amount.
Output (region, customer_id, customer_rank_in_region, order_id, order_rank_for_customer).
"""

In [0]:
orders_data = [
    ("O001", "C1", "2024-01-10", 500.0),
    ("O002", "C1", "2024-02-15", 300.0),
    ("O003", "C2", "2024-01-20", 700.0),
    ("O004", "C2", "2024-03-05", 200.0),
    ("O005", "C3", "2024-01-12", 900.0),
    ("O006", "C3", "2024-02-10", 100.0),
    ("O007", "C4", "2024-03-01", 800.0),
    ("O008", "C5", "2024-03-15", 400.0),
    ("O009", "C6", "2024-01-25", 950.0),
    ("O010", "C6", "2024-02-18", 450.0),
    ("O011", "C7", "2024-01-30", 600.0),
    ("O012", "C8", "2024-02-22", 750.0),
    ("O013", "C9", "2024-03-07", 550.0),
]

regions_data = [
    ("C1", "East"),
    ("C2", "East"),
    ("C3", "East"),
    ("C4", "West"),
    ("C5", "West"),
    ("C6", "North"),
    ("C7", "North"),
    ("C8", "North"),
    ("C9", "South"),
]

# Orders schema (date as string initially)
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),   # keep as string first
    StructField("order_amount", DoubleType(), True)
])

orders_df_raw = spark.createDataFrame(orders_data, schema=orders_schema)

# Regions schema
regions_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("region", StringType(), True)
])

regions_df = spark.createDataFrame(regions_data, schema=regions_schema)

orders_df = orders_df_raw.withColumn("order_date", to_date("order_date", "yyyy-MM-dd"))

orders_df.show()
regions_df.show()


In [0]:
# For each region, find the top 5 customers by total order amount.

# Joining both customers orders and regions tables to get region details for customers
joined_df = orders_df.join(regions_df, on="customer_id", how="inner")
joined_df.show()

# Aggregating total orders amount for each customer in region
cust_agg_df = joined_df.groupBy("customer_id", "region").agg(
    sum("order_amount").alias("total_orders_amount")
)
cust_agg_df.show()

# Ranking customers per region
window_spec = Window.partitionBy("region").orderBy(desc("total_orders_amount"))

cust_ranking_df = cust_agg_df.withColumn(
    "customer_rank_in_region",
    dense_rank().over(window_spec)
)

# Filtering top 5 customers per region
region_top5_cust_df = cust_ranking_df.filter(col("customer_rank_in_region") <= 5)

region_top5_cust_df.show()

# Joining filtered data with orders to get order details for top 5 customers
top5_cust_with_orders_df = region_top5_cust_df.join(orders_df, on="customer_id", how="inner")
top5_cust_with_orders_df.show()

# Top 5 customers orders ranking 
window_rnk = Window.partitionBy("customer_id").orderBy(desc("order_amount"))

top5_cust_orders_ranking_df = top5_cust_with_orders_df.withColumn(
    "order_rank_for_customer",
    dense_rank().over(window_rnk)
)

top5_cust_orders_ranking_df = top5_cust_orders_ranking_df.select(
    col("region"),
    col("customer_id"),
    col("customer_rank_in_region"),
    col("order_id"),
    col("order_rank_for_customer")
)

top5_cust_orders_ranking_df.show()

In [0]:
"""
Assignment 4 – Churn Prediction Data Prep
You have:
usage_logs: (customer_id, usage_date, minutes_used)
support_calls: (customer_id, call_date, issue_type)

Task:
For each customer, find the average daily usage over the last 90 days.
Count how many support calls they made in the last 180 days.
Create a final table (customer_id, avg_daily_usage_90d, support_calls_180d) that can be used for churn prediction.
"""

In [0]:
usage_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("usage_date", StringType(), True),   # string first
    StructField("minutes_used", IntegerType(), True)
])

usage_data = [
    ("C001", "2024-06-01", 30),
    ("C001", "2024-06-05", 45),
    ("C001", "2024-08-25", 20),
    ("C002", "2024-07-10", 60),
    ("C002", "2024-08-01", 40),
    ("C002", "2024-08-20", 50),
    ("C003", "2024-05-15", 25),
    ("C003", "2024-08-10", 35),
]

usage_logs = spark.createDataFrame(usage_data, schema=usage_schema)

# convert string to actual DateType
usage_logs = usage_logs.withColumn("usage_date", to_date("usage_date", "yyyy-MM-dd"))

usage_logs.show()

calls_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("call_date", StringType(), True),   # string first
    StructField("issue_type", StringType(), True)
])

calls_data = [
    ("C001", "2024-07-01", "billing"),
    ("C001", "2024-08-15", "technical"),
    ("C002", "2024-06-20", "technical"),
    ("C002", "2024-07-25", "account"),
    ("C003", "2024-08-05", "billing"),
]

support_calls = spark.createDataFrame(calls_data, schema=calls_schema)

# convert string to actual DateType
support_calls = support_calls.withColumn("call_date", to_date("call_date", "yyyy-MM-dd"))

support_calls.show()


In [0]:
class ChurnPrediction:
    def read_data(self, path):
        try:
            df = spark.read.format("csv").option("header", True).load(path)
            return df
        except Exception as err:
            print(f"Error caught {err}")

    def churnAnalysis(self, usage_logs_df, support_calls_df):
        # Filtering last 90 days data from usage logs    
        usage_logs_df = usage_logs_df.filter(col("usage_date") >= date_sub(current_date(), 90))   

        # Filtering last 180 days from support calls 
        support_calls_df = support_calls_df.filter(col("call_date") > date_sub(current_date(), 180))

        # Aggregating total daily usage for customers
        cust_agg_df = usage_logs_df.groupBy("customer_id").agg(
            sum("minutes_used").alias("total_minutes"),
            count("usage_date").alias("active_days")
        )

        customers_daily_usage_df = cust_agg_df.withColumn(
            "avg_daily_usage_90d",
            col("total_minutes") / col("active_days")
        )

        # Counting total support calls for last 180 days
        cust_total_support_calls_df = support_calls_df.groupBy("customer_id").agg(
            count("call_date").alias("support_calls_180d")
        )

        # Joining customers both usage and support calls 
        cust_with_usage_support_calls_df = customers_daily_usage_df.join(cust_total_support_calls_df, on="customer_id", how="left")

        # Selecting required columns from joined dataframe
        cust_with_usage_support_calls_df = cust_with_usage_support_calls_df.select(
            col("customer_id"),
            col("avg_daily_usage_90d"),
            col("support_calls_180d")
        )

        return cust_with_usage_support_calls_df
    

# Instantiate the class
churn_inst = ChurnPrediction()

# Reading files from path
usage_logs_df = churn_inst.read_data("/Volumes/db_prac/prac/usage_logs")
support_calls_df = churn_inst.read_data("/Volumes/db_prac/prac/support_calls")

# Reading the methods
cust_with_usage_support_calls_df = churn_inst.churnAnalysis(usage_logs_df, support_calls_df)

# Showing dataframes
cust_with_usage_support_calls_df.show()

In [0]:
users_data = [
    ("U1", "New York"),
    ("U2", "New York"),
    ("U3", "Los Angeles"),
    ("U4", "Los Angeles"),
    ("U5", "Chicago"),
]

users_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("city", StringType(), True)
])

users_df = spark.createDataFrame(users_data, schema=users_schema)
users_df.show()

calls_data = [
    ("C1", "U1", "U2", "2023-01-01 09:00:00", "2023-01-01 09:10:00"),
    ("C2", "U2", "U3", "2023-01-01 09:05:00", "2023-01-01 09:20:00"),
    ("C3", "U1", "U3", "2023-01-02 10:00:00", "2023-01-02 10:05:00"),
    ("C4", "U3", "U4", "2023-01-02 10:01:00", "2023-01-02 10:07:00"),
    ("C5", "U1", "U5", "2023-01-02 10:03:00", "2023-01-02 10:06:00"),  # overlap for suspicious pattern
    ("C6", "U2", "U1", "2023-01-03 15:00:00", "2023-01-03 15:30:00"),
    ("C7", "U4", "U5", "2023-01-03 15:05:00", "2023-01-03 15:25:00"),
    ("C8", "U1", "U2", "2023-01-03 15:07:00", "2023-01-03 15:12:00"),  # another suspicious candidate
    ("C9", "U5", "U1", "2023-01-07 08:00:00", "2023-01-07 08:10:00"),
    ("C10", "U1", "U3", "2023-01-07 08:03:00", "2023-01-07 08:05:00"), # suspicious pattern
]

calls_schema = StructType([
    StructField("call_id", StringType(), True),
    StructField("caller_id", StringType(), True),
    StructField("receiver_id", StringType(), True),
    StructField("call_start", StringType(), True),  # string first
    StructField("call_end", StringType(), True)     # string first
])

calls_df = spark.createDataFrame(calls_data, schema=calls_schema)

calls_df = calls_df.withColumn("call_start", to_timestamp("call_start")) \
    .withColumn("call_end", to_timestamp("call_end"))
calls_df.show()



In [0]:
"""
Assignment 9 – E-commerce Multi-Order Analysis

Datasets:
orders(order_id, customer_id, order_date, total_amount)
order_items(order_id, product_id, quantity, price)
customers(customer_id, region, signup_date)

Tasks:
For each order, calculate number of distinct products and total quantity.
For each customer, compute:
First order date.
Average gap (days) between consecutive orders (use lag).
For each region, find the top 3 repeat customers by number of orders
"""

In [0]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", StringType(), False),
    StructField("total_amount", DoubleType(), False)
])

orders_data = [
    (1, 101, "2023-01-05", 250.0),
    (2, 101, "2023-02-10", 300.0),
    (3, 102, "2023-01-15", 150.0),
    (4, 103, "2023-01-20", 400.0),
    (5, 101, "2023-03-01", 200.0),
    (6, 104, "2023-02-25", 500.0),
    (7, 102, "2023-03-10", 220.0),
    (8, 103, "2023-03-15", 180.0),
    (9, 105, "2023-03-18", 300.0)
]

orders_df = spark.createDataFrame(orders_data, schema=orders_schema)
orders_df = orders_df.withColumn("order_date", to_date("order_date"))
orders_df.show()


order_items_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("price", DoubleType(), False)
])

order_items_data = [
    (1, 201, 2, 50.0),
    (1, 202, 3, 50.0),
    (2, 201, 1, 100.0),
    (2, 203, 2, 100.0),
    (3, 204, 1, 150.0),
    (4, 202, 4, 100.0),
    (4, 205, 2, 100.0),
    (5, 206, 5, 40.0),
    (6, 207, 2, 250.0),
    (7, 201, 2, 110.0),
    (8, 204, 3, 60.0),
    (9, 208, 1, 300.0)
]

order_items_df = spark.createDataFrame(order_items_data, schema=order_items_schema)
order_items_df.show()

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("region", StringType(), False),
    StructField("signup_date", StringType(), False)
])

customers_data = [
    (101, "North", "2022-12-15"),
    (102, "South", "2023-01-01"),
    (103, "East", "2023-01-10"),
    (104, "West", "2023-02-01"),
    (105, "North", "2023-03-01")
]

customers_df = spark.createDataFrame(customers_data, schema=customers_schema)
customers_df = customers_df.withColumn("signup_date", to_date("signup_date"))
customers_df.show()

In [0]:
# For each order, calculate number of distinct products and total quantity.
orders_distinct_products_total_quantity_df = order_items_df.groupBy("order_id").agg(
    countDistinct("product_id").alias("total_distinct_products"),
    sum("quantity").alias("total_quantity")
)

orders_distinct_products_total_quantity_df.show()

# For each customer, compute: First order date, Average gap (days) between consecutive orders (use lag).
customers_firstOrders_df = orders_df.groupBy("customer_id").agg(
    min("order_date").alias("first_order_date")
) 

window_spec = Window.partitionBy("customer_id").orderBy("order_date")

customers_orders_prevdetails_df = orders_df.withColumn(
    "prev_order_date",
    lag("order_date").over(window_spec)
)

customers_orders_diff_df = customers_orders_prevdetails_df.withColumn(
    "day_diff",
    datediff("order_date", "prev_order_date")
)

customers_avg_gap_df = customers_orders_diff_df.groupBy("customer_id").agg(
    avg("day_diff").alias("avg_gap")
)
customers_avg_gap_df.show()

# For each region, find the top 3 repeat customers by number of orders.
customers_with_regions_df = customers_df.join(orders_df, on="customer_id", how="inner") 

customers_total_orders_df = customers_with_regions_df.groupBy("customer_id", "region").agg(
    count("order_id").alias("total_orders")
)

window_rnk = Window.partitionBy("region").orderBy(desc("total_orders"))

customers_ranking_in_regions_df = customers_total_orders_df.withColumn(
    "cust_rnk",
    dense_rank().over(window_rnk)
)

# Filtering top 3 customers per region
top3_customers_df = customers_ranking_in_regions_df.filter(col("cust_rnk") <= 3)
top3_customers_df.show()

In [0]:
"""
Assignment 11 Bank Loan Risk Analysis
Datasets:
loans(loan_id, customer_id, loan_amount, loan_date, status)

payments(payment_id, loan_id, payment_date, payment_amount)

customers(customer_id, city, income)

Tasks:
For each loan, compute outstanding balance = loan_amount – sum(payments).
For each customer, calculate their loan default rate (% of loans with status = "default").
Rank cities by average default rate.
Find customers with 2+ consecutive missed payments (window with lag).
"""

In [0]:
# ---------------- Customers ----------------
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("income", DoubleType(), True)
])

customers_data = [
    ("C1", "Delhi", 80000),
    ("C2", "Delhi", 60000),
    ("C3", "Mumbai", 75000),
    ("C4", "Bangalore", 50000),
    ("C5", "Bangalore", 90000)
]

customers_df = spark.createDataFrame(customers_data, customers_schema)
customers_df.show()

# ---------------- Loans ----------------
loans_schema = StructType([
    StructField("loan_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("loan_amount", DoubleType(), True),
    StructField("loan_date", StringType(), True),
    StructField("status", StringType(), True)
])

loans_data = [
    ("L1", "C1", 10000, "2024-01-01", "active"),
    ("L2", "C1", 15000, "2024-03-01", "default"),
    ("L3", "C2", 20000, "2024-02-15", "default"),
    ("L4", "C3", 12000, "2024-04-01", "active"),
    ("L5", "C4", 18000, "2024-01-20", "default"),
    ("L6", "C5", 25000, "2024-02-25", "active")
]

loans_df = spark.createDataFrame(loans_data, loans_schema)
loans_df.withColumn("loan_date", to_date("loan_date"))
loans_df.show()

# ---------------- Payments ----------------
payments_schema = StructType([
    StructField("payment_id", StringType(), True),
    StructField("loan_id", StringType(), True),
    StructField("payment_date", StringType(), True),
    StructField("payment_amount", DoubleType(), True)
])

payments_data = [
    ("P1", "L1", "2024-01-15", 2000),
    ("P2", "L1", "2024-02-15", 3000),
    ("P3", "L2", "2024-03-20", 5000),
    ("P4", "L3", "2024-03-10", 8000),
    ("P5", "L3", "2024-04-10", 2000),
    # L4 has no payments → outstanding = full loan
    ("P6", "L5", "2024-02-01", 4000),
    # L5 has missed payments after partial
    ("P7", "L6", "2024-03-01", 10000),
    ("P8", "L6", "2024-03-15", 5000),
    # Some missed payments for L6 too
]

payments_df = spark.createDataFrame(payments_data, payments_schema)
payments_df = payments_df.withColumn("payment_date", to_date("payment_date"))
payments_df.show()



In [0]:
# For each loan, compute outstanding balance = loan_amount – sum(payments).
# Aggregating total payments for each loan
total_payments_df = payments_df.groupBy("loan_id").agg(
    sum("payment_amount").alias("loan_total_payment")
)

# Join with loans to calculate outstanding amount
loans_updated_df = loans_df.join(total_payments_df, on="loan_id", how="inner")

# Calculating outstanding amount
loans_outstanding_amounts_df = loans_updated_df.withColumn(
    "outstanding_amount",
    col("loan_amount") - col("loan_total_payment")
)
loans_outstanding_amounts_df.show()
"""-------------------------"""
# For each customer, calculate their loan default rate (% of loans with status = "default").
# Customers totalloans and default loans
cust_loans_details_df =  loans_df.groupBy("customer_id").agg(
    count("loan_id").alias("total_loans"),
    count(when(col("status") == "default", True)).alias("default_loans")
)

# Customers default rate
cust_default_rates_df = cust_loans_details_df.withColumn(
    "default_rate",
    round(col("default_loans") / col("total_loans") * 100, 2) 
)
cust_default_rates_df.show()
"""-------------------------"""
# Rank cities by average default rate.
# Joining with customers dataset to get customer city details
cust_with_cities_df = cust_default_rates_df.join(customers_df, on="customer_id", how="inner")

# Aggregating average default rates for cities
city_avg_default_rates_df = cust_with_cities_df.groupBy("city").agg(
    avg("default_rate").alias("avg_default_rate")
)

# Ranking cities based on avg default rate 
window_rnk = Window.orderBy(desc("avg_default_rate"))

ranking_cities_df = city_avg_default_rates_df.withColumn(
    "city_rnk",
    dense_rank().over(window_rnk)
)
ranking_cities_df.show()

# Find customers with 2+ consecutive missed payments (window with lag).
# Joining payments with loans to get customer details
upd_df = loans_df.alias("l1").join(payments_df.alias("p1"), on="loan_id", how="inner")

# Selecting required columns
cust_loans_df = upd_df.select(
    col("l1.customer_id").alias("customer_id"),
    col("p1.loan_id").alias("loan_id"),
    col("p1.payment_date").alias("payment_date"),
    col("p1.payment_amount").alias("payment_amount")
)

# Window specification for getting previous payment amount details 
window_spec = Window.partitionBy("customer_id", "loan_id").orderBy("payment_date")
customers_payments_ordering_df = cust_loans_df.withColumn(
    "prev_payment",
    lag("payment_amount").over(window_spec)
)

# Flagging loans where current and previous payments are 0
flagging_df = customers_payments_ordering_df.withColumn(
    "consec_miss_flag",
    when((col("payment_amount") == 0) & (col("prev_payment") == 0), 1).otherwise(0)
)

# Aggregating total missed payments
loans_missed_payments_df = flagging_df.groupBy("customer_id", "loan_id").agg(
    sum("consec_miss_flag").alias("total_missed_payments")
)

# Filtering customers
filtered_df = loans_missed_payments_df.select("customer_id").filter(col("total_missed_payments") >= 1).distinct()

filtered_df.show()

In [0]:
"""
Assignment 15 – Music Streaming Top Charts

Datasets:
plays(play_id, user_id, song_id, play_time, duration_played)
songs(song_id, artist_id, song_length)
artists(artist_id, artist_name)

Tasks:
For each song, calculate play completion rate = avg(duration_played / song_length).
Rank songs by popularity per week (unique users who played).
For each artist, find their most popular song per month.
Identify “loyal listeners”: users who listened to the same artist at least 10 times across 3 consecutive weeks.
"""

In [0]:
# ----------------------------
# songs dataframe
# ----------------------------
songs_schema = StructType([
    StructField("song_id", IntegerType(), True),
    StructField("artist_id", IntegerType(), True),
    StructField("song_length", DoubleType(), True)  # in seconds
])

songs_data = [
    (1, 101, 180.0),
    (2, 102, 210.0),
    (3, 101, 200.0),
    (4, 103, 240.0),
    (5, 104, 150.0)
]

songs_df = spark.createDataFrame(songs_data, schema=songs_schema)
songs_df.show()

# ----------------------------
# artists dataframe
# ----------------------------
artists_schema = StructType([
    StructField("artist_id", IntegerType(), True),
    StructField("artist_name", StringType(), True)
])

artists_data = [
    (101, "Artist A"),
    (102, "Artist B"),
    (103, "Artist C"),
    (104, "Artist D")
]

artists_df = spark.createDataFrame(artists_data, schema=artists_schema)
artists_df.show()

# ----------------------------
# plays dataframe
# ----------------------------
plays_schema = StructType([
    StructField("play_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("song_id", IntegerType(), True),
    StructField("play_time", StringType(), True),
    StructField("duration_played", DoubleType(), True)  # in seconds
])

plays_data = [
    (1001, 1, 1, "2025-09-01 10:00:00", 180.0),
    (1002, 1, 2, "2025-09-02 12:00:00", 200.0),
    (1003, 2, 1, "2025-09-03 14:00:00", 170.0),
    (1004, 2, 3, "2025-09-05 16:00:00", 180.0),
    (1005, 3, 4, "2025-09-07 18:00:00", 240.0),
    (1006, 3, 5, "2025-09-10 20:00:00", 150.0),
    (1007, 1, 3, "2025-09-11 09:00:00", 200.0),
    (1008, 1, 1, "2025-09-15 10:00:00", 180.0),
    (1009, 2, 2, "2025-09-16 11:00:00", 210.0),
    (1010, 3, 1, "2025-09-18 15:00:00", 180.0)
]

# Create DataFrame with play_time as string
plays_df = spark.createDataFrame(plays_data, plays_schema)

# Convert play_time column to timestamp
plays_df = plays_df.withColumn("play_time", to_timestamp(col("play_time"), "yyyy-MM-dd HH:mm:ss"))

plays_df.show(truncate=False)

In [0]:
# For each song, calculate play completion rate = avg(duration_played / song_length)

# Joining both plays and songs on song_id 
songs_with_lengths_df = plays_df.join(songs_df, on="song_id", how="inner")
songs_with_lengths_df.show()

songs_completion_rates_df = songs_with_lengths_df.groupBy("song_id").agg(
    avg(col("duration_played") / col("song_length")).alias("song_completion_rate")
) 
songs_completion_rates_df.show()
"----------------------------------------------------"
# Rank songs by popularity per week (unique users who played).
updated_df = plays_df.withColumn("week", date_trunc("week", col("play_time")))

# Aggregating total unique users per week
song_unique_users_df = updated_df.groupBy("week", "song_id").agg(
    countDistinct("user_id").alias("total_unique_users")
)

# Ranking songs per week based on total users count
window_spec = Window.partitionBy("week").orderBy(desc("total_unique_users"))

ranking_songs_df = song_unique_users_df.withColumn(
    "song_rnk",
    dense_rank().over(window_spec)
)
ranking_songs_df.show()

"--------------------------------------------------"

# For each artist, find their most popular song per month.

# Aggregating total users count for each song for each artist on each month
songs_with_lengths_df = songs_with_lengths_df.withColumn(
    "year_month",
    date_format("play_time", "yyyy_MM")
)
aggregated_df = songs_with_lengths_df.groupBy("artist_id", "song_id", "year_month").agg(
    countDistinct("user_id").alias("total_users")
)

# Ranking songs for each artist based on count
window_rnk = Window.partitionBy("artist_id", "year_month").orderBy(desc("total_users"))

ranking_artist_songs_df = aggregated_df.withColumn(
    "song_rnk",
    dense_rank().over(window_rnk)
)

# Filter most popular song per month for artists
artist_popular_songs_df = ranking_artist_songs_df.filter(col("song_rnk") == 1)
artist_popular_songs_df.show()

# Identify “loyal listeners”: users who listened to the same artist at least 10 times across 3 consecutive weeks.

# Add a week column (start of week)
plays_df = plays_df.withColumn("week", date_trunc("week", col("play_time")))

# Join plays with songs to get artist_id
plays_artist_df = plays_df.join(songs_df, on="song_id", how="inner")

# Aggregate song count per user per artist per week
user_artist_weekly = plays_artist_df.groupBy("user_id", "artist_id", "week") \
    .count("song_id") \
    .withColumnRenamed("count", "weekly_song_count")

# 1. Get distinct users and artists
users = plays_df.select("user_id").distinct()
artists = songs_df.select("artist_id").distinct()

# 2. Get all weeks
weeks = plays_df.select("week").distinct()

# Cross join users and artists
user_artist = users.crossJoin(artists)

# Cross join with weeks to get full timeline
full_timeline = user_artist.crossJoin(weeks)

# 4. Left join actual plays counts
full_user_artist_week = full_timeline.join(user_artist_weekly, 
                                           on=["user_id", "artist_id", "week"], 
                                           how="left") \
                                    .fillna(0, subset=["weekly_song_count"])


# Define rolling window: current week + 2 preceding weeks
w = Window.partitionBy("user_id", "artist_id").orderBy("week").rowsBetween(-2, 0)

# Calculate rolling 3-week sum
full_user_artist_week = full_user_artist_week.withColumn(
    "three_week_sum",
    sum("weekly_song_count").over(w)
)

# Flagging users who listened to the same artist at least 10 times across 3 consecutive weeks.
full_user_artist_week = full_user_artist_week.withColumn(
    "flag",
    when(col("three_week_sum") >= 10, "Loyal_listener").otherwise("Not_loyal")
)

# Filtering oyal listeners
loyal_listeners_df = full_user_artist_week.filter(col("flag") == "Loyal_listener")
loyal_listeners_df.show()

In [0]:
# -------------------- timesheets (string dates) --------------------
timesheets_data = [
    (101, "2025-09-01", 8, "P001"),
    (101, "2025-09-02", 9, "P001"),
    (101, "2025-09-03", 10, "P002"),
    (101, "2025-09-04", 8, "P002"),
    (101, "2025-09-05", 12, "P001"),  # Total 47 hrs

    (102, "2025-09-01", 7, "P001"),
    (102, "2025-09-02", 7, "P001"),
    (102, "2025-09-03", 8, "P003"),
    (102, "2025-09-04", 7, "P003"),
    (102, "2025-09-05", 8, "P001"),   # Total 37 hrs

    (103, "2025-09-01", 10, "P002"),
    (103, "2025-09-02", 9, "P002"),
    (103, "2025-09-03", 9, "P004"),
    (103, "2025-09-04", 8, "P004"),
    (103, "2025-09-05", 10, "P002"),  # Total 46 hrs
]

timesheets_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("work_date", StringType(), True),
    StructField("hours_worked", DoubleType(), True),
    StructField("project_id", StringType(), True)
])

timesheets_df = spark.createDataFrame(timesheets_data, schema=timesheets_schema)

# Convert string → date
timesheets_df = timesheets_df.withColumn("work_date", to_date("work_date", "yyyy-MM-dd"))
timesheets_df.show()

# -------------------- projects --------------------
projects_data = [
    ("P001", "Migration", 10),
    ("P002", "Automation", 20),
    ("P003", "Analytics", 10),
    ("P004", "AI Research", 30)
]

projects_schema = StructType([
    StructField("project_id", StringType(), True),
    StructField("project_name", StringType(), True),
    StructField("dept_id", IntegerType(), True)
])

projects_df = spark.createDataFrame(projects_data, schema=projects_schema)
projects_df.show()

# -------------------- employees (string dates) --------------------
employees_data = [
    (101, 10, "2020-05-15"),
    (102, 10, "2022-02-10"),
    (103, 20, "2019-11-01")
]

employees_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("dept_id", IntegerType(), True),
    StructField("hire_date", StringType(), True)
])

employees_df = spark.createDataFrame(employees_data, schema=employees_schema)

# Convert string → date
employees_df = employees_df.withColumn("hire_date", to_date("hire_date", "yyyy-MM-dd"))
employees_df.show()


In [0]:
timesheets_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/db_prac/prac/raw/timesheets")

employees_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/db_prac/prac/raw/emps")

projects_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/db_prac/prac/raw/projects")


In [0]:
"""
Assignment 13 – Retail Inventory Turnover

Datasets:
inventory(product_id, store_id, stock_date, stock_level)
sales(order_id, product_id, store_id, sale_date, quantity)

Tasks:
Compute daily stock balance per product-store.
Join sales and inventory to detect stockout events (sales > stock).
For each product, compute inventory turnover ratio = total quantity sold / average stock (use window for avg).
Rank stores within each region by turnover ratio.
"""

In [0]:
# Inventory dataset
inventory_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("stock_date", StringType(), True),
    StructField("stock_level", IntegerType(), True)
])

inventory_data = [
    (101, 1, "2025-10-01", 120),
    (101, 1, "2025-10-02", 100),
    (101, 1, "2025-10-03", 60),
    (102, 1, "2025-10-01", 200),
    (102, 1, "2025-10-02", 180),
    (103, 2, "2025-10-01", 150),
    (103, 2, "2025-10-02", 100),
    (104, 3, "2025-10-01", 80),
    (104, 3, "2025-10-02", 50)
]

inventory_df = spark.createDataFrame(inventory_data, schema=inventory_schema)
inventory_df = inventory_df.withColumn("stock_date", to_date("stock_date"))
inventory_df.show()

# Sales dataset
sales_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("quantity", IntegerType(), True)
])

sales_data = [
    (1001, 101, 1, "2025-10-01", 30),
    (1002, 101, 1, "2025-10-02", 50),
    (1003, 101, 1, "2025-10-03", 70),
    (1004, 102, 1, "2025-10-02", 40),
    (1005, 103, 2, "2025-10-02", 60),
    (1006, 104, 3, "2025-10-02", 70)
]

sales_df = spark.createDataFrame(sales_data, schema=sales_schema)
sales_df = sales_df.withColumn("sale_date", to_date("sale_date"))
sales_df.show()

# Region mapping (for final ranking)
region_data = [
    (1, "South"),
    (2, "North"),
    (3, "East")
]
region_schema = StructType([
    StructField("store_id", IntegerType(), True),
    StructField("region", StringType(), True)
])
region_df = spark.createDataFrame(region_data, schema=region_schema)
region_df.show()




In [0]:
# Compute daily stock balance per product-store.

# Joining both inventory and sales to get stock details
product_stocks_df = inventory_df.alias("inv").join(
    sales_df.alias("sal"),
     (col("inv.product_id") == col("sal.product_id") &
      col("inv.store_id") == col("sal.store_id") &
      col("inv.stock_date") == col("sal.sale_date")),
     "left").fillna(0)

# Checking stock balance after sales for products
product_bal_df = product_stocks_df.withColumn(
    "stock_balance",
    col("quantity") - col("stock_level")
)     

# Aggregating total stock balances per product daily
product_agg_df = product_bal_df.groupBy("product_id", "store_id", "stock_date").agg(
    sum("stock_balance").alias("daily_stock_balances")
)
products_agg_df.show()


# Join sales and inventory to detect stockout events (sales > stock).
product_stockouts_df = product_stocks_df.filter(col("quantity") > col("stock_level"))
product_stockouts_df.show()

# For each product, compute inventory turnover ratio = total quantity sold / average stock (use window for avg).

# Aggregating total quantity sold and average stock per product
product_total_sold_df = sales_df.groupBy("product_id", "store_id").agg(
    sum("quantity").alias("total_quantity_sold")
)

product_avg_stock_df = inventory_df.groupBy("product_id", "store_id").agg(
    avg("stock_level").alias("avg_stock")
)

product_with_total_avg_metrics_df = product_total_sold_df.join(product_avg_stock_df, on=["product_id", "store_id"], "inner")

# Calculating turnover ratio for products
products_turnover_ratio_df = product_with_total_avg_metrics_df.withColumn(
    "turnover_ratio",
    col("total_quantity_sold") / col("avg_stock")
)
products_turnover_ratio_df.show()

# Rank stores within each region by turnover ratio.

# Joining with regions to get region_id
products_regions_df = products_turnover_ratio_df.join(region_df, on="store_id", how="inner")

# Aggregating average turnover ratio per store
store_agg_df = products_regions_df.groupBy("store_id", "region").agg(
    avg("turnover_ratio").alias("avg_turnover_ratio")
)

# Window specification for ranking stores 
window_spec = Window.partitionBy("region").orderBy(desc("avg_turnover_ratio"))

ranking_stores_df = store_agg_df.withColumn(
    "store_rnk",
    dense_rank().over(window_spec)
)
ranking_stores_df.orderBy(asc("store_rnk")).show()

In [0]:
flights_schema = StructType([
    StructField("flight_id", StringType(), True),
    StructField("airline", StringType(), True),
    StructField("dep_airport", StringType(), True),
    StructField("arr_airport", StringType(), True),
    StructField("dep_time", StringType(), True),   # initially as string
    StructField("arr_time", StringType(), True),   # initially as string
    StructField("delay_minutes", IntegerType(), True)
])

flights_data = [
    ("F001", "A1", "JFK", "LAX", "2025-01-05 08:00:00", "2025-01-05 11:00:00", 45),
    ("F002", "A1", "JFK", "ORD", "2025-01-15 09:30:00", "2025-01-15 11:00:00", 10),
    ("F003", "A2", "LAX", "SEA", "2025-02-10 06:00:00", "2025-02-10 08:00:00", 70),
    ("F004", "A2", "SEA", "DEN", "2025-02-20 14:00:00", "2025-02-20 17:00:00", 55),
    ("F005", "A3", "ORD", "DFW", "2025-03-01 07:00:00", "2025-03-01 09:00:00", 20),
    ("F006", "A1", "LAX", "ORD", "2025-03-15 12:00:00", "2025-03-15 17:00:00", 90),
    ("F007", "A2", "JFK", "LAX", "2025-03-25 10:00:00", "2025-03-25 14:00:00", 60),
    ("F008", "A3", "DFW", "SEA", "2025-04-02 11:00:00", "2025-04-02 13:00:00", 35),
    ("F009", "A1", "ORD", "JFK", "2025-04-05 09:00:00", "2025-04-05 13:00:00", 55),
    ("F010", "A2", "SEA", "DEN", "2025-04-12 15:00:00", "2025-04-12 18:00:00", 20),
    ("F011", "A1", "LAX", "JFK", "2025-05-03 06:00:00", "2025-05-03 12:00:00", 80),
    ("F012", "A3", "ORD", "LAX", "2025-05-08 14:00:00", "2025-05-08 18:00:00", 25),
    ("F013", "A1", "JFK", "LAX", "2025-06-02 08:00:00", "2025-06-02 12:00:00", 65),
    ("F014", "A2", "LAX", "SEA", "2025-06-10 07:00:00", "2025-06-10 09:00:00", 40),
    ("F015", "A3", "DFW", "ORD", "2025-06-20 10:00:00", "2025-06-20 13:00:00", 10)
]

flights_df = spark.createDataFrame(flights_data, schema=flights_schema)
flights_df.display()
flights_df.write.format("csv").mode("overwrite").option("header", True).save("/Volumes/db_prac/prac/raw/flights_data")

flights_df = (
    flights_df
    .withColumn("dep_time", to_timestamp("dep_time", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("arr_time", to_timestamp("arr_time", "yyyy-MM-dd HH:mm:ss"))
)

airlines_schema = StructType([
    StructField("airline_id", StringType(), True),
    StructField("airline_name", StringType(), True)
])

airlines_data = [
    ("A1", "SkyJet Airways"),
    ("A2", "Pacific Air"),
    ("A3", "BlueWings Airlines")
]

airlines_df = spark.createDataFrame(airlines_data, schema=airlines_schema)

airlines_df.display()
airlines_df.write.mode("overwrite").format("csv").option("header", True).save("/Volumes/db_prac/prac/raw/airlines_data")

In [0]:
class AirlineDelayTracker:
    def read_data(self, path):
        try:
            df = spark.read.format("csv").option("header", True).load(path)
            return df 
        except Exception as err:
            print(f"Error occured while reading file{err}")
            return None
        
    def airlineAvgDelays(self, flights_df):
        # Creating month column from dep_time
        flights_df = flights_df.withColumn("month", date_trunc("month", col("dep_time")))

        # Aggregating average delay for each airline per month
        airline_avg_delays_df = flights_df.groupBy("airline", "month").agg(
            avg("delay_minutes").alias("monthly_avg_delay")
        )     
        return airline_avg_delays_df
    

    def flightsRanking(self, flights_df):
        # Creating month column from dep_time
        flights_df = flights_df.withColumn("month", date_trunc("month", col("dep_time")))

        # Window specification for ranking airports for airlines
        window_spec = Window.partitionBy("airline", "month").orderBy(desc("delay_minutes"))

        # Ranking flights in airlines based on delay in minutes
        flights_rankings_df = flights_df.withColumn(
            "flight_rnk",
            dense_rank().over(window_spec)
        )

        # Filtering top3 flights per airline
        top3_flights_df = flights_rankings_df.filter(col("flight_rnk") <= 3)

        return top3_flights_df
    
    def mostDelayedAirports(self, flights_df):
        # Creating month and airport columns in flights_df
        flights_upd_df = (
            flights_df
            .withColumn("month", date_trunc("month", col("dep_time"))) 
            .withColumn("airport", col("dep_airport"))
        )

        # Aggregating monthly avg delays per month for airports
        airports_monthly_avg_delay_df = flights_upd_df.groupBy("airport", "month").agg(
            avg("delay_minutes").alias("current_month_delay")
        )

        # Window specification to get previous month avg delays for airports
        window_lag = Window.partitionBy("airport").orderBy("month")

        # Checking previous months averages
        airports_avg_data_df = airports_monthly_avg_delay_df.withColumn(
            "prev_month_delay",
            lag("current_month_delay").over(window_lag)
        )

        airports_checking_df = airports_avg_data_df.withColumn(
            "streak",
            when(col("prev_month_delay").isNull() | ((col("current_month_delay")>= 30) & (col("prev_month_delay") >= 30)), 1).otherwise(0)
        )

        # Aggregating streaks of threshold for airports
        aggregated_df = airports_checking_df.groupBy("airport").agg(
            sum("streak").alias("total_months")
        )

        # Filtering airports 
        filtered_df = aggregated_df.filter(col("total_months") >= 3)

        delayed_airports_df = filtered_df.select(col("airport"))

        return delayed_airports_df

# Instantiating the class
airport_inst = AirlineDelayTracker()

# Reading file from path
flights_df= airport_inst.read_data("/Volumes/db_prac/prac/raw/flights_data")

# Reading methods
airline_avg_delays_df = airport_inst.airlineAvgDelays(flights_df)
top3_flights_df = airport_inst.flightsRanking(flights_df)
delayed_airports_df = airport_inst.mostDelayedAirports(flights_df)
